In [ ]:
import bs4
import lxml
import pandas
import urllib
import time

from urllib import request

In [ ]:
!pip install playwright
!playwright install

!playwright install chromium
!playwright install-deps

!apt-get update
!apt-get install -y libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libxkbcommon0 \
    libxdamage1 libxrandr2 libasound2 libpangocairo-1.0-0 libpango-1.0-0 libxcomposite1 \
    libxfixes3 libxrender1 libxcursor1 libgbm1 libatspi2.0-0 libwayland-client0 \
    fonts-liberation libappindicator3-1 libdbusmenu-gtk3-4 libxss1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 19.4 MB/s eta 0:00:00
(node:946) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [] 0% 0.0s164.7 MiB [] 0% 40.7s164.7 MiB [] 0% 12.4s164.7 MiB [] 0% 13.1s164.7 MiB [] 0% 6.6s164.7 MiB [] 2% 4.0s164.7 MiB [] 3% 3.0s164.7 MiB [] 4% 2.8s164.7 MiB [] 5% 2.5s164.7 MiB [] 6% 2.3s164.7 MiB [] 7% 2.1s164.7 MiB [] 8% 2.1s164.7 MiB [] 9% 2.0s164.7 MiB [] 10% 1.9s164.7 MiB [] 11% 1.8s164.7 MiB [] 12% 1.8s164.7 MiB [] 14% 1.7s164.7 MiB [] 15% 1.7s164.7 MiB [] 15% 1.6s164.7 MiB [] 16% 1.6s164.7 MiB [] 18% 1.6s164.7 MiB [] 19% 1.5s164.7 MiB [] 20% 1.5s164.7 MiB [] 21% 1.5s164.7 MiB [] 22% 1.4s164.7 MiB [] 23% 1.4s164.7 MiB [] 25% 1.3s164.7 MiB [] 26% 1.3s164.7 MiB [] 27% 1.3s164.7 MiB [] 28% 1.3s16

In [ ]:
#CHARGER LE FICHIER restaurants.csv qui contient tous les liens vers des restaurants
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
#TELECHARGER LE FICHIER
from google.colab import files
files.download("restaurants.csv")


In [ ]:
#nom_restaurant == 0                  représente le nom du restaurant
#adresse == 1                         représente l'adresse du restaurant
#code_postale == 2                    représente le code postale de l'adresse
#nom_plat == 3                        représente le nom du plat
#prix_plat == 4                       représente le prix du plat donné par le restaurant
#description == 5                     représente la liste des ingrédients du plat
#categorie_plat == 6                  représente la section dans laquelle le restaurant présente le plat (ex : McChicken dans la categorie burger)
#prix_brut == 7                       représente le prix du plat en comptant seulement les ingrédients. Le prix des ingrédients est celui trouvé dans des supermarchés (sera déduit avec une autre base de donnée)
#note_restaurant == 8                 représente la note du restaurant, [OPTIONEL]
#nombre_avis == 9                     représente le nombre de personne ayant notée le restaurant, [OPTIONEL]
#spécialité_restaurant == 10          représente la spécialité du restaurant (ex sur UberEats le menu déroulant possède plusieurs catégorie : pizzas, sushis, burgers, dessert)
#autres_spécialités_restaurant == 11  représente les autres spécialités du restaurant (s'arrête à informations)
#categorie_prix_restaurant == 12      représente le prix moyen du restaurant (€ : moins chère, €€ : moyennnement chère ...), [OPTIONEL]
#taille_restaurant == 13              représente la taille du restaurant en m^2, sera remplit si on a le temps

In [ ]:
#On récolte la liste de tous les liens d'une catégorie de restaurant, puis on copie-colle la liste pour la sauvegarder.
#Ensuite il ne faut pas oublier de modifier la case 10 : "spécialité_restaurant". Elle prend la valeur de la catégorie actuelle des liens (ex : Burger, Pizzas, Sushis)
#Pour finir on lance l'algorithme dans la deuxième case avec au maximum 10 restaurants de la liste de liens obtenu avant

############CREER UNE CONDITION QUI EMPECHE D'AVOIR DEUX FOIS LE MEME RESTAURANT (une liste qui contient tous les noms de restaurant)

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import asyncio


async def main():
    #  |BURGERS|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Burgers"
    #  |PIZZAS|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Pizza"
    #  |SUSHIS|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Sushi"
    #  |POKE|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Poke"
    #  |AILES DE POULET|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Wings"
    #  |SANDWICH|url ="https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Sandwich"
    #  |SOUPES|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Soup"
    #  |ITALIEN|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Italian"
    #  |DESSERT|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Dessert"
    #  |MEXICAIN|url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D&scq=Mexican"
    url = "https://www.ubereats.com/fr/feed?diningMode=DELIVERY&pl=JTdCJTIyYWRkcmVzcyUyMiUzQSUyMlBhcmlzJTIyJTJDJTIycmVmZXJlbmNlJTIyJTNBJTIyQ2hJSlBTdEkwQ1Z1NWtjUkFGSFk0Y2FDQ3h3JTIyJTJDJTIycmVmZXJlbmNlVHlwZSUyMiUzQSUyMmdvb2dsZV9wbGFjZXMlMjIlMkMlMjJsYXRpdHVkZSUyMiUzQTQ4Ljg2NDA0OTMlMkMlMjJsb25naXR1ZGUlMjIlM0EyLjMzMTA1MjYlN0Q%3D"
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url)
        await page.wait_for_timeout(5000)  # laisser le temps à la page de charger

        # Clique sur les boutons "Afficher plus"
        while True:
            try:
                button = await page.query_selector("button:has-text('Afficher plus')")
                if button:
                    print(" Clic sur 'Afficher plus' ...")
                    await button.click()
                    await page.wait_for_timeout(3000)
                else:
                    break
            except Exception:
                break

        # Scrolle vers le bas de la page
        print(" Scroll de la page...")
        previous_height = 0
        for _ in range(15):
            await page.evaluate("window.scrollBy(0, document.body.scrollHeight)")
            await page.wait_for_timeout(2500)
            new_height = await page.evaluate("document.body.scrollHeight")
            if new_height == previous_height:
                break
            previous_height = new_height

        # Extraire le HTML et parser avec BeautifulSoup
        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")

        restaurant_cards = soup.find_all("a", attrs={"data-testid": "store-card"})
        print(f" {len(restaurant_cards)} restaurants trouvés après chargement complet.")

        base_url = "https://www.ubereats.com"
        restaurants = []

        #Les restaurants sont présents dans des balises h3
        for card in restaurant_cards:
            link = card.get("href")
            nom_tag = card.find("h3")
            nom = nom_tag.get_text(strip=True) if nom_tag else "Nom inconnu"

            if link:
                full_link = base_url + link
                restaurants.append({"nom": nom, "url": full_link})

        # Afiche les 15 premiers liens vers des restaurants
        for r in restaurants[:15]:
            print(f" {r['nom']} → {r['url']}")

        print(f"\n✅ Total collecté : {len(restaurants)} restaurants.")

        print(restaurants)
        await browser.close()


await main()


 Clic sur 'Afficher plus' ...
 Clic sur 'Afficher plus' ...
 Clic sur 'Afficher plus' ...
 Clic sur 'Afficher plus' ...
 Clic sur 'Afficher plus' ...
⬇ Scroll de la page...
 750 restaurants trouvés après chargement complet.
 McDonald's® → https://www.ubereats.com/fr/store/mcdonalds-paris-louvre-rivoli/nWnUwvnuRBWFIRsBjSXlEQ
 Pizza Primavera 🍕 → https://www.ubereats.com/fr/store/pizza-primavera-paris/HGHkqohYQ16XA7B2UhxuGA
 K-ViBE → https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA
 DongNe Chicken → https://www.ubereats.com/fr/store/dongne-chicken/KQtl8gnrXYi1eG_cSl7GEQ
 Street Bangkok Odéon → https://www.ubereats.com/fr/store/street-bangkok-odeon/g5aEKN46SX26a3zEnz_Yvg
 Momidji → https://www.ubereats.com/fr/store/momidji/DDIiwB7HRgiS_X96vUN7Rw
 Krousty Sabaïdi → https://www.ubereats.com/fr/store/krousty-sabaidi-chatelet/2_5Q6WfmUKKb523t2wXP4A
 O'Tacos → https://www.ubereats.com/fr/store/otacos-chatelet/iI439XtiRfSffGqy-aUJRw
 Chipotle Mexican Gr

In [ ]:
#On double clic sur le résultat du code précèdent et on l'affecte à liste_des_restaurants_dans_la_categorie_actuelle
liste_des_restaurants_dans_la_categorie_actuelle = [{'nom': "McDonald's®", 'url': 'https://www.ubereats.com/fr/store/mcdonalds-paris-louvre-rivoli/nWnUwvnuRBWFIRsBjSXlEQ'}, {'nom': 'Pizza Primavera 🍕', 'url': 'https://www.ubereats.com/fr/store/pizza-primavera-paris/HGHkqohYQ16XA7B2UhxuGA'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA'}, {'nom': 'DongNe Chicken', 'url': 'https://www.ubereats.com/fr/store/dongne-chicken/KQtl8gnrXYi1eG_cSl7GEQ'}, {'nom': 'Street Bangkok Odéon', 'url': 'https://www.ubereats.com/fr/store/street-bangkok-odeon/g5aEKN46SX26a3zEnz_Yvg'}, {'nom': 'Momidji', 'url': 'https://www.ubereats.com/fr/store/momidji/DDIiwB7HRgiS_X96vUN7Rw'}, {'nom': 'Krousty Sabaïdi', 'url': 'https://www.ubereats.com/fr/store/krousty-sabaidi-chatelet/2_5Q6WfmUKKb523t2wXP4A'}, {'nom': "O'Tacos", 'url': 'https://www.ubereats.com/fr/store/otacos-chatelet/iI439XtiRfSffGqy-aUJRw'}, {'nom': 'Chipotle Mexican Grill', 'url': 'https://www.ubereats.com/fr/store/chipotle-mexican-grill-montmartre/mR9vNHQNTa6RY2P5tKOjcQ'}, {'nom': 'KFC', 'url': 'https://www.ubereats.com/fr/store/kfc-boulevard-montmartre/44wIpktfW-6R5EkZ-7dm8w'}, {'nom': 'Homer Lobster 🦞', 'url': 'https://www.ubereats.com/fr/store/homer-lobster-odeon/iGu9uMFJXU68zj04YokaHQ'}, {'nom': 'Capri Saint-Honoré', 'url': 'https://www.ubereats.com/fr/store/capri-saint-honore/kUKEn1dbQqSnLNU-FJMDTQ'}, {'nom': 'MELT DELI', 'url': 'https://www.ubereats.com/fr/store/melt-deli-les-halles/OtaDJ1_oUhOxObUQfxvM6w'}, {'nom': 'Big Fernand', 'url': 'https://www.ubereats.com/fr/store/big-fernand-saint-honore/_vgdLZuFTXa1zxYQ-YMKLQ'}, {'nom': 'PITA Paris Opéra 🍋', 'url': 'https://www.ubereats.com/fr/store/pita-paris-opera/eNipQwNlUx28hUMAM0kIBg'}, {'nom': 'Kisoro', 'url': 'https://www.ubereats.com/fr/store/kisoro-montmartre/XFb_QPTCVqmZWzVhwiKyYA'}, {'nom': 'Pokawa', 'url': 'https://www.ubereats.com/fr/store/pokawa-chateaudun/bfBN-02RQ36iNsdr4SNfow'}, {'nom': 'Blend Hamburger', 'url': 'https://www.ubereats.com/fr/store/blend-hamburger-madeleine/cey-XJ0MTfyJYAKfNMV-fA'}, {'nom': 'Fuchill 🇭🇰', 'url': 'https://www.ubereats.com/fr/store/fuchill/RADSimphXg2tuoyWnSxL8g'}, {'nom': 'NOLA 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/nola-fried-chicken-louvre/5NoE6L5iXyGGFG6HiwdtlA'}, {'nom': 'Tacos & Fries', 'url': 'https://www.ubereats.com/fr/store/tacos-%26-fries-paris/0rinNWW7WHWp6Txhx5OAuw'}, {'nom': 'CIRCUS 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/circus-fried-chicken-louvre/TMxVYItSXNmrBQBDZ9fg7w'}, {'nom': 'Istanbul Charonne', 'url': 'https://www.ubereats.com/fr/store/istanbul-charonne/aqEOokA6Q46SPSEXkzOerw'}, {'nom': 'Sushi Shop', 'url': 'https://www.ubereats.com/fr/store/sushi-shop-reaumur/YDZ8iFZsTXel9-DsQFsUNA'}, {'nom': 'Pepe Chicken By FastGoodCuisine', 'url': 'https://www.ubereats.com/fr/store/pepe-chicken-by-fastgoodcuisine-grands-boulevards/jKwuK0BxVq-xg1jBBO4-tA'}, {'nom': 'Neko Ramen 🍜', 'url': 'https://www.ubereats.com/fr/store/neko-ramen/GT8lBKdJT-mrf5U1WQgbgg'}, {'nom': 'Pizza Hut', 'url': 'https://www.ubereats.com/fr/store/pizza-hut-paris-9-cadet/PBYDjz7rT4OTMRfy3M0kFg'}, {'nom': "Domino's Pizza", 'url': 'https://www.ubereats.com/fr/store/dominos-pizza-paris-2e/p6R3XK53QN-z3Q-LO6P2gg'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA'}, {'nom': 'Don yoku', 'url': 'https://www.ubereats.com/fr/store/don-yoku/pQuhqpbLS8ONQ2K10syQQA'}, {'nom': 'Trois Fois Plus de Piment', 'url': 'https://www.ubereats.com/fr/store/trois-fois-plus-de-piment-saint-martin/nzcLzXHnRlKl2nCobsfpxw'}, {'nom': 'Kisoro', 'url': 'https://www.ubereats.com/fr/store/kisoro-montmartre/XFb_QPTCVqmZWzVhwiKyYA'}, {'nom': 'Cô My Cantine', 'url': 'https://www.ubereats.com/fr/store/co-my-cantine/Kachu8IcTten2cwgR_sJiw'}, {'nom': 'Intermarché', 'url': 'https://www.ubereats.com/fr/store/intermarche-sebastopol/NtaHRJGzWeCm7G_bZufCjA?diningMode=DELIVERY'}, {'nom': 'Carrefour', 'url': 'https://www.ubereats.com/fr/store/carrefour-paris-richelieu-43/esDV0no9TuqkdkvAnpwCUg?diningMode=DELIVERY'}, {'nom': 'Monoprix', 'url': 'https://www.ubereats.com/fr/store/monoprix-opera/UL7BS3b4U4ezLfF1HWgxCQ?diningMode=DELIVERY'}, {'nom': 'Franprix', 'url': 'https://www.ubereats.com/fr/store/franprix-paris-marengo/9FMoclE6Rsu5ja6i0rZTcA?diningMode=DELIVERY'}, {'nom': 'Picard', 'url': 'https://www.ubereats.com/fr/store/picard-paris-bourse/cjC3S1nQWx6Q5rZXTRgMCA?diningMode=DELIVERY'}, {'nom': 'Carrefour XL', 'url': 'https://www.ubereats.com/fr/store/carrefour-xl-market-maubeuge/TOM_rARNRjSNwMDL4zLofQ?diningMode=DELIVERY'}, {'nom': "Monop'", 'url': 'https://www.ubereats.com/fr/store/monop-madeleine/jCFiRd_8QBaeOu1SzFfbjA?diningMode=DELIVERY'}, {'nom': "Bio c' Bon", 'url': 'https://www.ubereats.com/fr/store/bio-c-bon-paris-saint-augustin/8YpK9VSMVyK5oP8unABO6w?diningMode=DELIVERY'}, {'nom': 'U Express', 'url': 'https://www.ubereats.com/fr/store/u-express-montorgueil/HnFsHgNvVpa6hhuVFM4ghg?diningMode=DELIVERY'}, {'nom': 'La Belle Vie', 'url': 'https://www.ubereats.com/fr/store/la-belle-vie-paris/3EJWfVsRWx-dEyF6xm45dA?diningMode=DELIVERY'}, {'nom': 'Blueberry Maki Bar', 'url': 'https://www.ubereats.com/fr/store/blueberry-maki-bar/_QE-aGENWWi62eg_WUVsWw'}, {'nom': 'On the Bab', 'url': 'https://www.ubereats.com/fr/store/on-the-bab/5_BEcw8fUx2QSzGcjlV7Rw'}, {'nom': 'Petit Bao Faubourg Saint Denis', 'url': 'https://www.ubereats.com/fr/store/petit-bao-faubourg-saint-denis/tpJbG1a5WLCWLhPhztwDWA'}, {'nom': 'Takuto Handroll', 'url': 'https://www.ubereats.com/fr/store/takuto-handroll/E8aGw3g1UBGVlrid7dpnZA'}, {'nom': 'Fuchill 🇭🇰', 'url': 'https://www.ubereats.com/fr/store/fuchill/RADSimphXg2tuoyWnSxL8g'}, {'nom': 'Nari Nari 🍙 Cuisine Japonaise', 'url': 'https://www.ubereats.com/fr/store/nari-nari-cuisine-japonaise/IsM6pw6RUBa69n7ozLmG8Q'}, {'nom': 'Tiger Tiger by Panda Panda', 'url': 'https://www.ubereats.com/fr/store/tiger-tiger-by-panda-panda/NOTwhHXBTli5COm-FxbnzA'}, {'nom': 'Meokja 🇰🇷', 'url': 'https://www.ubereats.com/fr/store/meokja/12t82XF2Viu2uy30WGW1eA'}, {'nom': 'Suzie Wong by Panda Panda', 'url': 'https://www.ubereats.com/fr/store/suzie-wong-by-panda-panda/TyX-AdJ8WLagpa7__WyDhg'}, {'nom': 'Neko Ramen 🍜', 'url': 'https://www.ubereats.com/fr/store/neko-ramen/GT8lBKdJT-mrf5U1WQgbgg'}, {'nom': 'Le Cambodge', 'url': 'https://www.ubereats.com/fr/store/le-cambodge/A6mWcFVdRvyhOk04btP0cg'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA'}, {'nom': 'Raviolis du Sichuan 龙抄手', 'url': 'https://www.ubereats.com/fr/store/raviolis-du-sichuan-龙抄手-chatelet/_NqVEuaaWymfAeAMa3c7-w'}, {'nom': 'CHEZ QIN', 'url': 'https://www.ubereats.com/fr/store/chez-qin/4RTdwZdYXOunhOoNQaj2fw'}, {'nom': 'Benihana Paris', 'url': 'https://www.ubereats.com/fr/store/benihana-paris/gFZvbauYVbGsOlpSSfwPKA'}, {'nom': 'MELT Oberkampf', 'url': 'https://www.ubereats.com/fr/store/melt-oberkampf/EykFfNGbXx2lrtq4IG8kdw'}, {'nom': "L'As du Shawarma", 'url': 'https://www.ubereats.com/fr/store/las-du-shawarma-paris-12/lS71zxmIU2WmsSELK9kDTw'}, {'nom': 'Doki Doki', 'url': 'https://www.ubereats.com/fr/store/doki-doki-louvre/ZhsyMHROQj-0iw0xHBIYkw'}, {'nom': 'WANTED 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/wanted-smash-burgers-louvre/ufN0pyyiWXuog_wJpDevmA'}, {'nom': 'OH MY CHEESE! 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/oh-my-cheese!-smash-burgers-louvre/WgBhgihUXZSHjXH5SbmgJg'}, {'nom': 'JUICY 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/juicy-smash-burgers-louvre/U79oQ_K2VHGcKJgo2FwyLA'}, {'nom': "S'MORE 🍪 American Desserts", 'url': 'https://www.ubereats.com/fr/store/smore-american-desserts-louvre/Lsyejam4VWmE7oAIiKfHTA'}, {'nom': 'NOLA 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/nola-fried-chicken-louvre/5NoE6L5iXyGGFG6HiwdtlA'}, {'nom': 'Tacos & Fries', 'url': 'https://www.ubereats.com/fr/store/tacos-%26-fries-paris/0rinNWW7WHWp6Txhx5OAuw'}, {'nom': 'CIRCUS 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/circus-fried-chicken-louvre/TMxVYItSXNmrBQBDZ9fg7w'}, {'nom': 'SEOULMATE 🐔 🇰🇷 Korean Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/seoulmate-korean-fried-chicken-louvre/9uffSqSdVC2e6QjE5y8viw'}, {'nom': "Domino's Pizza", 'url': 'https://www.ubereats.com/fr/store/dominos-pizza-paris-2e/p6R3XK53QN-z3Q-LO6P2gg'}, {'nom': 'Bolo Bolo 🥟 Cuisine de Hong Kong', 'url': 'https://www.ubereats.com/fr/store/bolo-bolo-cuisine-de-hong-kong/y_OKFnufWdKyhVaDxXvc1Q'}, {'nom': 'In Korea', 'url': 'https://www.ubereats.com/fr/store/in-korea/7SPTcZANVbC2MUgF6PWuhQ'}, {'nom': 'Kisoro Sushi', 'url': 'https://www.ubereats.com/fr/store/kisoro-sushi-louvre/KHH7i819UeaOAL3XrwaLBw'}, {'nom': 'Petit Bao Faubourg Saint Denis', 'url': 'https://www.ubereats.com/fr/store/petit-bao-faubourg-saint-denis/tpJbG1a5WLCWLhPhztwDWA'}, {'nom': 'Ippudo Louvre', 'url': 'https://www.ubereats.com/fr/store/ippudo-louvre/DJ8bD_dsXlOPksDbIep-Jg'}, {'nom': 'Isa Sushi', 'url': 'https://www.ubereats.com/fr/store/isa-sushi/nOhoCuXsVVmvC6IcfGJeow'}, {'nom': 'JOAYO', 'url': 'https://www.ubereats.com/fr/store/joayo-luxembourg/rjN2fOsqQaSyAv1z75l98g'}, {'nom': 'Planet Sushi', 'url': 'https://www.ubereats.com/fr/store/planet-sushi-malesherbes/hIHViu8YVL-PBKCHln_80g'}, {'nom': 'Sushi Shop', 'url': 'https://www.ubereats.com/fr/store/sushi-shop-reaumur/YDZ8iFZsTXel9-DsQFsUNA'}, {'nom': 'KARE Curry & Donburi', 'url': 'https://www.ubereats.com/fr/store/kare-curry-%26-donburi/sQuzjZ9FUvysdp2D4TwHAQ'}, {'nom': 'NOLA 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/nola-fried-chicken-louvre/5NoE6L5iXyGGFG6HiwdtlA'}, {'nom': 'SEOULMATE 🐔 🇰🇷 Korean Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/seoulmate-korean-fried-chicken-louvre/9uffSqSdVC2e6QjE5y8viw'}, {'nom': "Domino's Pizza", 'url': 'https://www.ubereats.com/fr/store/dominos-pizza-paris-2e/p6R3XK53QN-z3Q-LO6P2gg'}, {'nom': 'In Korea', 'url': 'https://www.ubereats.com/fr/store/in-korea/7SPTcZANVbC2MUgF6PWuhQ'}, {'nom': 'WANTED 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/wanted-smash-burgers-louvre/ufN0pyyiWXuog_wJpDevmA'}, {'nom': 'OH MY CHEESE! 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/oh-my-cheese!-smash-burgers-louvre/WgBhgihUXZSHjXH5SbmgJg'}, {'nom': 'JUICY 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/juicy-smash-burgers-louvre/U79oQ_K2VHGcKJgo2FwyLA'}, {'nom': "S'MORE 🍪 American Desserts", 'url': 'https://www.ubereats.com/fr/store/smore-american-desserts-louvre/Lsyejam4VWmE7oAIiKfHTA'}, {'nom': 'Tacos & Fries', 'url': 'https://www.ubereats.com/fr/store/tacos-%26-fries-paris/0rinNWW7WHWp6Txhx5OAuw'}, {'nom': 'CIRCUS 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/circus-fried-chicken-louvre/TMxVYItSXNmrBQBDZ9fg7w'}, {'nom': 'Bolo Bolo 🥟 Cuisine de Hong Kong', 'url': 'https://www.ubereats.com/fr/store/bolo-bolo-cuisine-de-hong-kong/y_OKFnufWdKyhVaDxXvc1Q'}, {'nom': 'Petit Bao Faubourg Saint Denis', 'url': 'https://www.ubereats.com/fr/store/petit-bao-faubourg-saint-denis/tpJbG1a5WLCWLhPhztwDWA'}, {'nom': 'JOAYO', 'url': 'https://www.ubereats.com/fr/store/joayo-luxembourg/rjN2fOsqQaSyAv1z75l98g'}, {'nom': 'KARE Curry & Donburi', 'url': 'https://www.ubereats.com/fr/store/kare-curry-%26-donburi/sQuzjZ9FUvysdp2D4TwHAQ'}, {'nom': 'Isa Sushi', 'url': 'https://www.ubereats.com/fr/store/isa-sushi/nOhoCuXsVVmvC6IcfGJeow'}, {'nom': 'Le Cambodge', 'url': 'https://www.ubereats.com/fr/store/le-cambodge/A6mWcFVdRvyhOk04btP0cg'}, {'nom': 'Ippudo Louvre', 'url': 'https://www.ubereats.com/fr/store/ippudo-louvre/DJ8bD_dsXlOPksDbIep-Jg'}, {'nom': 'DimDimSum 🥟', 'url': 'https://www.ubereats.com/fr/store/dimdimsum/3U-povV-UxOnsGjCTS--QQ'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA'}, {'nom': 'Tiger Tiger by Panda Panda', 'url': 'https://www.ubereats.com/fr/store/tiger-tiger-by-panda-panda/NOTwhHXBTli5COm-FxbnzA'}, {'nom': "Le Parfum De L'Inde 🇮🇳", 'url': 'https://www.ubereats.com/fr/store/le-parfum-de-linde/HdRudXXQQRK4ym5mzRsPqg'}, {'nom': 'JOAYO', 'url': 'https://www.ubereats.com/fr/store/joayo-luxembourg/rjN2fOsqQaSyAv1z75l98g'}, {'nom': 'Street Bangkok Odéon', 'url': 'https://www.ubereats.com/fr/store/street-bangkok-odeon/g5aEKN46SX26a3zEnz_Yvg'}, {'nom': 'Mashil', 'url': 'https://www.ubereats.com/fr/store/mashil/GH7Id05oQdyc1e5kl1-Ncg'}, {'nom': 'Les Amis des Messina', 'url': 'https://www.ubereats.com/fr/store/les-amis-des-messina-reaumur/6V3ogZufSvm4giQY1AHs8w'}, {'nom': "Le Saigon D'antan", 'url': 'https://www.ubereats.com/fr/store/le-saigon-dantan/8JIlWRIITlmHjPNrCmtoJw'}, {'nom': 'Don yoku', 'url': 'https://www.ubereats.com/fr/store/don-yoku/pQuhqpbLS8ONQ2K10syQQA'}, {'nom': 'Royal Kashmir', 'url': 'https://www.ubereats.com/fr/store/royal-kashmir/TgcHTC9cQVyZ5SwZeztvNQ'}, {'nom': 'La Créole', 'url': 'https://www.ubereats.com/fr/store/la-creole/-k1KTzNFT6uW34F_XwYBuA'}, {'nom': 'Mao Dumpling Bar', 'url': 'https://www.ubereats.com/fr/store/mao-dumpling-bar/8ondF-SGS0i-XFBlTf5k5Q'}, {'nom': 'La Marmite de Boeuf', 'url': 'https://www.ubereats.com/fr/store/la-marmite-de-boeuf/tDxkOJHbU4q-COzeByXxsg'}, {'nom': 'Mme Shawn Caffarelli', 'url': 'https://www.ubereats.com/fr/store/mme-shawn-caffarelli/2rNKXuMtWcirw5IcUCC06w'}, {'nom': 'Chez Bebert', 'url': 'https://www.ubereats.com/fr/store/chez-bebert/55WIdcXLQUmvEyAQu15qMw'}, {'nom': 'Blend Hamburger', 'url': 'https://www.ubereats.com/fr/store/blend-hamburger-madeleine/cey-XJ0MTfyJYAKfNMV-fA'}, {'nom': 'Palais Royal Hong Kong', 'url': 'https://www.ubereats.com/fr/store/palais-royal-hong-kong/JZ1R1bHZTMq5ZaW1y5TROA'}, {'nom': 'Malibu African Food', 'url': 'https://www.ubereats.com/fr/store/malibu-african-food/puDOFeqITQG5-0j3K6A62g'}, {'nom': 'Vapiano', 'url': 'https://www.ubereats.com/fr/store/vapiano-champs-elysees/m7niXGZvROO4RinvPKe6WQ'}, {'nom': 'Assanabel', 'url': 'https://www.ubereats.com/fr/store/assanabel-alesia/g-Z0Z1k9RGu09rKsCQU9Hw'}, {'nom': 'Le Thali Royal', 'url': 'https://www.ubereats.com/fr/store/le-thali-royal/EvfyJJ29U-2N4Vw7VES55w'}, {'nom': 'Kisoro Sushi', 'url': 'https://www.ubereats.com/fr/store/kisoro-sushi-louvre/KHH7i819UeaOAL3XrwaLBw'}, {'nom': 'Pizza Primavera 🍕', 'url': 'https://www.ubereats.com/fr/store/pizza-primavera-paris/HGHkqohYQ16XA7B2UhxuGA'}, {'nom': 'Planet Sushi', 'url': 'https://www.ubereats.com/fr/store/planet-sushi-malesherbes/hIHViu8YVL-PBKCHln_80g'}, {'nom': "Domino's Pizza", 'url': 'https://www.ubereats.com/fr/store/dominos-pizza-paris-2e/p6R3XK53QN-z3Q-LO6P2gg'}, {'nom': 'Osaki Sushi', 'url': 'https://www.ubereats.com/fr/store/osaki-sushi-bonne-nouvelle/R0cvvgd2XBydGGlgzrNxgg'}, {'nom': 'Ippudo Louvre', 'url': 'https://www.ubereats.com/fr/store/ippudo-louvre/DJ8bD_dsXlOPksDbIep-Jg'}, {'nom': 'Sushi Shop', 'url': 'https://www.ubereats.com/fr/store/sushi-shop-reaumur/YDZ8iFZsTXel9-DsQFsUNA'}, {'nom': "Le Parfum De L'Inde 🇮🇳", 'url': 'https://www.ubereats.com/fr/store/le-parfum-de-linde/HdRudXXQQRK4ym5mzRsPqg'}, {'nom': 'In Korea', 'url': 'https://www.ubereats.com/fr/store/in-korea/7SPTcZANVbC2MUgF6PWuhQ'}, {'nom': 'Bolo Bolo 🥟 Cuisine de Hong Kong', 'url': 'https://www.ubereats.com/fr/store/bolo-bolo-cuisine-de-hong-kong/y_OKFnufWdKyhVaDxXvc1Q'}, {'nom': 'Neko Ramen 🍜', 'url': 'https://www.ubereats.com/fr/store/neko-ramen/GT8lBKdJT-mrf5U1WQgbgg'}, {'nom': 'Nari Nari 🍙 Cuisine Japonaise', 'url': 'https://www.ubereats.com/fr/store/nari-nari-cuisine-japonaise/IsM6pw6RUBa69n7ozLmG8Q'}, {'nom': 'Yatai Ramen', 'url': 'https://www.ubereats.com/fr/store/yatai-ramen-pyramide/DPIAIhoIV3-S53YlJ1ZudQ'}, {'nom': 'Popine', 'url': 'https://www.ubereats.com/fr/store/popine-2eme/DTwaaLgMTLScHVAHFayRPQ'}, {'nom': 'Petit Bao Faubourg Saint Denis', 'url': 'https://www.ubereats.com/fr/store/petit-bao-faubourg-saint-denis/tpJbG1a5WLCWLhPhztwDWA'}, {'nom': 'JOAYO', 'url': 'https://www.ubereats.com/fr/store/joayo-luxembourg/rjN2fOsqQaSyAv1z75l98g'}, {'nom': 'Boucherie Boudalia', 'url': 'https://www.ubereats.com/fr/store/boucherie-boudalia-halal-avs-paris/52MNnICuX0y_s07cQ4_yVg'}, {'nom': 'Dimitris Apéro Grec', 'url': 'https://www.ubereats.com/fr/store/dimitris-apero-grec-rambuteau/g6YnQzyAXRKIEo8jLXhvZw'}, {'nom': 'Bellota Bellota', 'url': 'https://www.ubereats.com/fr/store/bellota-bellota-champs-elysees/rssQ56COS_Kn5MnpYZSc9w'}, {'nom': 'Primeur des Ternes', 'url': 'https://www.ubereats.com/fr/store/primeur-des-ternes/IM8tlPMNWoyl-3DqWAPmrA'}, {'nom': 'Häagen-Dazs', 'url': 'https://www.ubereats.com/fr/store/haagen-dazs-opera-paris/s2IEC6VGWHeEl7Yi4xs-iA'}, {'nom': 'Delitaly 17ème', 'url': 'https://www.ubereats.com/fr/store/delitaly-17eme/V4MxGMbWVBevhrcakWOFFw'}, {'nom': 'Apero Kif', 'url': 'https://www.ubereats.com/fr/store/apero-kif-apero-tapas-et-finger-food-8eme/u6d99Ie2S2691CD_0IAwxw'}, {'nom': 'World Candy Shop 🤤', 'url': 'https://www.ubereats.com/fr/store/world-candy-shop-paris/Vpbcbv7RXT2W0xFl_-Widw'}, {'nom': 'Edwart Chocolatier', 'url': 'https://www.ubereats.com/fr/store/edwart-chocolatier-concorde/8Y6w2UWtX3OGs_meSxI3RQ'}, {'nom': 'Boucherie Husscut', 'url': 'https://www.ubereats.com/fr/store/boucherie-husscut-paris/C4s2Y28DXQyAZlNp0IfAcg'}, {'nom': 'Apertivus Saint Antoine', 'url': 'https://www.ubereats.com/fr/store/apertivus-saint-antoine-les-dieux-de-lapero/lwviB0z9VBW5-LZWW4i-nw'}, {'nom': 'Boucherie Halal de la Plaine AVS', 'url': 'https://www.ubereats.com/fr/store/boucherie-halal-de-la-plaine-avs/uC5owuWOU1SEfe3rdfZoKQ'}, {'nom': 'Mon Epicerie Paris  Vegan', 'url': 'https://www.ubereats.com/fr/store/mon-epicerie-paris-vegan/XNOVfaHzQOqrC_pVsDNKXA'}, {'nom': "Fédération Française de l'Apéritif", 'url': 'https://www.ubereats.com/fr/store/federation-francaise-de-laperitif-martyrs-paris/Vsalw9WhUBKDTBaIEK79Rg'}, {'nom': 'Primeur De Gama', 'url': 'https://www.ubereats.com/fr/store/primeur-de-gama/6cmts1BnUVSl696HsmtgiA'}, {'nom': 'Primeur Halles Convention', 'url': 'https://www.ubereats.com/fr/store/primeur-halles-convention/VaUxYgN9QwC8PNelYg8O5w'}, {'nom': 'Boucherie de la Place Halal', 'url': 'https://www.ubereats.com/fr/store/boucherie-de-la-place-halal/Hr5bo41pXWi66H0rjOAzdQ'}, {'nom': 'Boucherie La Ferme du Kremlin', 'url': 'https://www.ubereats.com/fr/store/boucherie-la-ferme-du-kremlin/Sf34Ya0FUcGDY26TDDCjww'}, {'nom': 'Dessert Factory', 'url': 'https://www.ubereats.com/fr/store/dessert-factory-paris-9e/ZodgoqbDRjmvcGjIDGZ-6w'}, {'nom': 'Primeur Halles Rambuteau', 'url': 'https://www.ubereats.com/fr/store/primeur-halles-rambuteau/h4JEFRZDRle0xTTUVnBiTw'}, {'nom': 'La Feuille des Aurès', 'url': 'https://www.ubereats.com/fr/store/la-feuille-des-aures-villejuif/qYKcwSBCVwuZb-WODwwSlw'}, {'nom': 'Glaces & Chocolats La Fabrique Givrée', 'url': 'https://www.ubereats.com/fr/store/glaces-%26-chocolats-la-fabrique-givree-paris/4nBavTjbTBWp32Zilrqhgg'}, {'nom': 'Prosciutti e Formaggi', 'url': 'https://www.ubereats.com/fr/store/prosciutti-e-formaggi/jsC4ZYH8XCWBkW7qME65UA'}, {'nom': 'Le Relais du Fruit Convention 15ème', 'url': 'https://www.ubereats.com/fr/store/le-relais-du-fruit-convention-15eme/7PLuaTjHU8-51AFAMSF1AA'}, {'nom': 'Epicerie Japonaise  Umami Matcha Café To Go', 'url': 'https://www.ubereats.com/fr/store/epicerie-japonaise-umami-matcha-cafe-to-go/9NggU08CVwaSFQrgSWMuTQ'}, {'nom': 'Touimer Bio', 'url': 'https://www.ubereats.com/fr/store/touimer-bio-primeur/95Aa5ZJSRc6Cw9_ehXuwOA'}, {'nom': 'La Ferme de Courbevoie', 'url': 'https://www.ubereats.com/fr/store/la-ferme-de-courbevoie-boucherie-halal/jjuuxaXYVpSrWg0KtslsMw'}, {'nom': 'VENCHI Montorgueil', 'url': 'https://www.ubereats.com/fr/store/venchi-montorgueil/gFZAc1E5VJSVPcky-BpBFw'}, {'nom': 'Amorino', 'url': 'https://www.ubereats.com/fr/store/amorino-opera/ePaMrbCuU-mnNlK7ldOsvQ'}, {'nom': 'La Boucherie de Courbevoie', 'url': 'https://www.ubereats.com/fr/store/la-boucherie-de-courbevoie/0KDla2yDUq6hDlwHtfSn8w'}, {'nom': 'Glaces Grom Express', 'url': 'https://www.ubereats.com/fr/store/glaces-grom-express-damremont/TLdU9a3JXz6aXvoK-I7uRQ'}, {'nom': 'Au Jardin Des Dames', 'url': 'https://www.ubereats.com/fr/store/au-jardin-des-dames-paris/c61b4iGTVWuQK3xsfQz4Jg'}, {'nom': 'Boucherie Eugen Lena', 'url': 'https://www.ubereats.com/fr/store/boucherie-eugen-lena-paris/k5SS1vQ-Wp24Hierk8oVKQ'}, {'nom': 'Rifai Victor Hugo', 'url': 'https://www.ubereats.com/fr/store/rifai-victor-hugo/RgJ1FFCVWn2ZUh0Q_dr27Q'}, {'nom': 'Rosée du Jardin', 'url': 'https://www.ubereats.com/fr/store/rosee-du-jardin/1ggTmxDRXv6ELS1d9Fttgg'}, {'nom': "Glaces Magnum x Ben & Jerry's", 'url': 'https://www.ubereats.com/fr/store/glaces-magnum-x-ben-%26-jerrys-champs-elysees/pkoL1ve0S2qahOhx1011_Q'}, {'nom': 'Primeur des 4 routes', 'url': 'https://www.ubereats.com/fr/store/primeur-des-4-routes/PG21LE4sSd214BV7gWHapA'}, {'nom': 'Les Halles Cambronne', 'url': 'https://www.ubereats.com/fr/store/les-halles-cambronne-paris/ocY3-ptXX4mJOE6RyuMLPQ'}, {'nom': 'Gosselin Primeurs', 'url': 'https://www.ubereats.com/fr/store/gosselin-primeurs-lourmel/VLyik7AlXYGavq_lBMBfLQ'}, {'nom': 'La Ferme De La Reine', 'url': 'https://www.ubereats.com/fr/store/la-ferme-de-la-reine/W1q74DSRW82Aq9GVpA1DCQ'}, {'nom': 'Les Jardins de Suresnes', 'url': 'https://www.ubereats.com/fr/store/les-jardins-de-suresnes/Y3p9uEDgU9afgXQFK1fELQ'}, {'nom': "Mon P'tit Panier Primeur", 'url': 'https://www.ubereats.com/fr/store/mon-ptit-panier-primeur-paris/pi8UpX7XXK2G_k_lcd8hdQ'}, {'nom': 'Poissonnerie Vent et Marée', 'url': 'https://www.ubereats.com/fr/store/poissonnerie-vent-et-maree-paris/fHFelTaIXGmPRAB5ou-6kw'}, {'nom': 'Churros à la maison', 'url': 'https://www.ubereats.com/fr/store/churros-a-la-maison/osCh_yfSVF-8vZk915qHkw'}, {'nom': 'Boucherie Moderne', 'url': 'https://www.ubereats.com/fr/store/boucherie-moderne-paris/hcbefpYFWjSDYEMLS6wdWQ'}, {'nom': 'Boucherie Leboeuf', 'url': 'https://www.ubereats.com/fr/store/boucherie-leboeuf-paris/RYpB49RVXOm22CnwDCe6Nw'}, {'nom': 'BeKef Boucherie', 'url': 'https://www.ubereats.com/fr/store/bekef-boucherie/lV0aiP9-QwioRd2XqTzUEQ'}, {'nom': 'Boucherie Casher Chez Victor Berbeche', 'url': 'https://www.ubereats.com/fr/store/boucherie-casher-chez-victor-berbeche-paris/vkjc-7wqW-iCVzyesUpUWg'}, {'nom': 'Carrefour', 'url': 'https://www.ubereats.com/fr/store/carrefour-paris-richelieu-43/esDV0no9TuqkdkvAnpwCUg'}, {'nom': 'Carrefour XL', 'url': 'https://www.ubereats.com/fr/store/carrefour-xl-market-pantin-avenue-jean-lolive-200/f3-XcsmOX-WIoQ-3xn9xoQ'}, {'nom': 'Carrefour Sprint', 'url': 'https://www.ubereats.com/fr/store/carrefour-sprint-paris-clichy-23/aC2bG4cvVvuKaOKEfKduIg'}, {'nom': 'U Express', 'url': 'https://www.ubereats.com/fr/store/u-express-petits-carreaux/Zqq3BL7hVCm6Uqs1KqLdyw'}, {'nom': 'Picard', 'url': 'https://www.ubereats.com/fr/store/picard-mazet/P18XSMS4WFuHyedN7bG-Qg'}, {'nom': 'Monop’', 'url': 'https://www.ubereats.com/fr/store/monop-paris-arago/JTXKKIq1VYap4WhLQXwPBg'}, {'nom': 'Franprix', 'url': 'https://www.ubereats.com/fr/store/franprix-paris-germain/xR448-JISBC_cgIxLAFTFg'}, {'nom': 'Cocci Market', 'url': 'https://www.ubereats.com/fr/store/cocci-market-bobigny/2efFHCJ2XPa_p5jYRnnOzA'}, {'nom': 'V24 Marché', 'url': 'https://www.ubereats.com/fr/store/v24-marche-paris/QWC8B9gRUVeEbQQBYJTUJQ'}, {'nom': 'Diago Ternes', 'url': 'https://www.ubereats.com/fr/store/diago-ternes-paris/q767agzZW96hYgCxnMd1kA'}, {'nom': 'Supermarché G20', 'url': 'https://www.ubereats.com/fr/store/supermarche-g20-marjolin/uwYAqeDcXDuGd3_E_Qqgkg'}, {'nom': 'Le Petit Marché Frichti', 'url': 'https://www.ubereats.com/fr/store/le-petit-marche-frichti-paris-berger-paris/zsr_bV9CXgy5X9hjJQRDJQ'}, {'nom': 'Kiyomart', 'url': 'https://www.ubereats.com/fr/store/kiyomart-paris/S_7k4mR1VuyNGyEoS0mElQ'}, {'nom': 'Sharu Mini Market', 'url': 'https://www.ubereats.com/fr/store/sharu-mini-market-paris/6fjSzRAbWS6Q9lkQX6aExg'}, {'nom': 'M.N Market', 'url': 'https://www.ubereats.com/fr/store/mn-market/w_Vi3RE_SZClY7G4_IzyhA'}, {'nom': 'Yummy So', 'url': 'https://www.ubereats.com/fr/store/yummy-so-epicerie-coreenne/M3LwyTBbX0G09L-FN49V0w'}, {'nom': 'Night Express', 'url': 'https://www.ubereats.com/fr/store/night-express-villejuif/DHzG3emzWn-Av9nbwvvRYg'}, {'nom': 'Marche Tout en 1 PROXI', 'url': 'https://www.ubereats.com/fr/store/marche-tout-en-1-proxi/_q6caubNTs64peq2f3RfJw'}, {'nom': 'Le Cambodge', 'url': 'https://www.ubereats.com/fr/store/le-cambodge/A6mWcFVdRvyhOk04btP0cg'}, {'nom': 'BMK Paris-Bamako', 'url': 'https://www.ubereats.com/fr/store/bmk-paris-bamako/Ht3Mp2R9TcWN3VJx5tXAtw'}, {'nom': 'Levantine', 'url': 'https://www.ubereats.com/fr/store/levantine/cM5-KvvjTT-m0HiLX99g1g'}, {'nom': 'Bambou Kitchen 🎋', 'url': 'https://www.ubereats.com/fr/store/bambou-kitchen/EdBObqJdWsOZLYjp7bMnxg'}, {'nom': 'Planet Sushi', 'url': 'https://www.ubereats.com/fr/store/planet-sushi-malesherbes/hIHViu8YVL-PBKCHln_80g'}, {'nom': 'OKAYA Sushi', 'url': 'https://www.ubereats.com/fr/store/okaya-sushi/aCFK7cCLXmqz0TIQguXc6Q'}, {'nom': 'Chipotle Mexican Grill', 'url': 'https://www.ubereats.com/fr/store/chipotle-mexican-grill-montmartre/mR9vNHQNTa6RY2P5tKOjcQ'}, {'nom': 'SUSHI Virgins', 'url': 'https://www.ubereats.com/fr/store/sushi-virgins/s0BXmBthSK6Z8ZA0MxX3Uw'}, {'nom': 'Pokawa', 'url': 'https://www.ubereats.com/fr/store/pokawa-chateaudun/bfBN-02RQ36iNsdr4SNfow'}, {'nom': 'Pho68', 'url': 'https://www.ubereats.com/fr/store/pho68/jWe_ZREWU1y66VeU9VFIAg'}, {'nom': 'Kapunka Cantine Thaï', 'url': 'https://www.ubereats.com/fr/store/kapunka-cantine-thai-montorgueil/79HsuLFAT3OmQqHOi7BF5Q'}, {'nom': 'Chez Georgette', 'url': 'https://www.ubereats.com/fr/store/chez-georgette/MjNFcSqeVJq6iDe8DdYQMA'}, {'nom': 'Groot la Tourte', 'url': 'https://www.ubereats.com/fr/store/groot-la-tourte/dEk06LhgXdGpUdoZG22nNw'}, {'nom': 'Extra Bowls 8ème – Bols chauds et froids', 'url': 'https://www.ubereats.com/fr/store/extra-bowls-8eme-–-bols-chauds-et-froids/4XoDtjn-X8CxXKPua_AKzg'}, {'nom': 'BARAK', 'url': 'https://www.ubereats.com/fr/store/barak-poissonniere/Ot_N-mDNQ3mUZPmjks9XQA'}, {'nom': 'Matsuri', 'url': 'https://www.ubereats.com/fr/store/matsuri-rue-du-bac/friCHm7fQfGfcMZlirE24w'}, {'nom': 'Meokja 🇰🇷', 'url': 'https://www.ubereats.com/fr/store/meokja/12t82XF2Viu2uy30WGW1eA'}, {'nom': 'Banh Mi Phuoc', 'url': 'https://www.ubereats.com/fr/store/banh-mi-phuoc/Bge6g5rZXHSRKmTk9SpDwA'}, {'nom': 'Hokkaido', 'url': 'https://www.ubereats.com/fr/store/hokkaido/hqG-9feMRcqQhUZfoE07_A'}, {'nom': 'Heiko Poké', 'url': 'https://www.ubereats.com/fr/store/heiko-poke-paris-lafayette/ic03ZDn0Wj6l8iFLz7NiLg'}, {'nom': 'Fuchill 🇭🇰', 'url': 'https://www.ubereats.com/fr/store/fuchill/RADSimphXg2tuoyWnSxL8g'}, {'nom': 'Miuko Thé', 'url': 'https://www.ubereats.com/fr/store/miuko-the/vd30Yft7RyOQNNASkeb1mQ'}, {'nom': 'Ça Croustille !', 'url': 'https://www.ubereats.com/fr/store/ca-croustille-!/WtF_NhuMX4iRIFXowOlCfw'}, {'nom': 'Blueberry Maki Bar', 'url': 'https://www.ubereats.com/fr/store/blueberry-maki-bar/_QE-aGENWWi62eg_WUVsWw'}, {'nom': 'Istanbul Charonne', 'url': 'https://www.ubereats.com/fr/store/istanbul-charonne/aqEOokA6Q46SPSEXkzOerw'}, {'nom': 'Isa Sushi', 'url': 'https://www.ubereats.com/fr/store/isa-sushi/nOhoCuXsVVmvC6IcfGJeow'}, {'nom': 'Kisoro Sushi', 'url': 'https://www.ubereats.com/fr/store/kisoro-sushi-louvre/KHH7i819UeaOAL3XrwaLBw'}, {'nom': 'Takuto Handroll', 'url': 'https://www.ubereats.com/fr/store/takuto-handroll/E8aGw3g1UBGVlrid7dpnZA'}, {'nom': 'On the Bab', 'url': 'https://www.ubereats.com/fr/store/on-the-bab/5_BEcw8fUx2QSzGcjlV7Rw'}, {'nom': 'Pizza Primavera 🍕', 'url': 'https://www.ubereats.com/fr/store/pizza-primavera-paris/HGHkqohYQ16XA7B2UhxuGA'}, {'nom': 'Planet Sushi', 'url': 'https://www.ubereats.com/fr/store/planet-sushi-malesherbes/hIHViu8YVL-PBKCHln_80g'}, {'nom': 'Osaki Sushi', 'url': 'https://www.ubereats.com/fr/store/osaki-sushi-bonne-nouvelle/R0cvvgd2XBydGGlgzrNxgg'}, {'nom': 'Ippudo Louvre', 'url': 'https://www.ubereats.com/fr/store/ippudo-louvre/DJ8bD_dsXlOPksDbIep-Jg'}, {'nom': "Le Parfum De L'Inde 🇮🇳", 'url': 'https://www.ubereats.com/fr/store/le-parfum-de-linde/HdRudXXQQRK4ym5mzRsPqg'}, {'nom': 'In Korea', 'url': 'https://www.ubereats.com/fr/store/in-korea/7SPTcZANVbC2MUgF6PWuhQ'}, {'nom': 'Bolo Bolo 🥟 Cuisine de Hong Kong', 'url': 'https://www.ubereats.com/fr/store/bolo-bolo-cuisine-de-hong-kong/y_OKFnufWdKyhVaDxXvc1Q'}, {'nom': 'Neko Ramen 🍜', 'url': 'https://www.ubereats.com/fr/store/neko-ramen/GT8lBKdJT-mrf5U1WQgbgg'}, {'nom': 'Petit Bao Faubourg Saint Denis', 'url': 'https://www.ubereats.com/fr/store/petit-bao-faubourg-saint-denis/tpJbG1a5WLCWLhPhztwDWA'}, {'nom': "Le Saigon D'antan", 'url': 'https://www.ubereats.com/fr/store/le-saigon-dantan/8JIlWRIITlmHjPNrCmtoJw'}, {'nom': 'JOAYO', 'url': 'https://www.ubereats.com/fr/store/joayo-luxembourg/rjN2fOsqQaSyAv1z75l98g'}, {'nom': 'Les Frères des Batignolles', 'url': 'https://www.ubereats.com/fr/store/les-freres-des-batignolles-paris-17/Obvfw642XSyu-KLur0akCw'}, {'nom': 'Naruto Ramen Montmartre 🍜', 'url': 'https://www.ubereats.com/fr/store/naruto-ramen-montmartre/rP7ggoYcToOgq4nVp91PVg'}, {'nom': 'BOLKIRI Street Food Viêt 🔥', 'url': 'https://www.ubereats.com/fr/store/bolkiri-street-food-viet-paris-11/pUF4HZiBUtm0Jt1L7UHwSw'}, {'nom': 'Kisoro Sushi', 'url': 'https://www.ubereats.com/fr/store/kisoro-sushi-louvre/KHH7i819UeaOAL3XrwaLBw'}, {'nom': 'La Pause Libanaise', 'url': 'https://www.ubereats.com/fr/store/la-pause-libanaise-saint-lazare/BMfWOIr1SQ-5cjWaJUvMgQ'}, {'nom': 'Hokkaido', 'url': 'https://www.ubereats.com/fr/store/hokkaido/hqG-9feMRcqQhUZfoE07_A'}, {'nom': 'Le Cambodge', 'url': 'https://www.ubereats.com/fr/store/le-cambodge/A6mWcFVdRvyhOk04btP0cg'}, {'nom': 'Okito', 'url': 'https://www.ubereats.com/fr/store/okito-montrouge/vW7bvrbWTUKu_Su8wa3yzw'}, {'nom': 'Kokoro Sushi', 'url': 'https://www.ubereats.com/fr/store/kokoro-sushi/oBinc55vRyCzH2ni5r6wZg'}, {'nom': 'Kisoro', 'url': 'https://www.ubereats.com/fr/store/kisoro-montmartre/XFb_QPTCVqmZWzVhwiKyYA'}, {'nom': 'OKAYA Sushi', 'url': 'https://www.ubereats.com/fr/store/okaya-sushi/aCFK7cCLXmqz0TIQguXc6Q'}, {'nom': 'Pizzeria Luigi Pepone', 'url': 'https://www.ubereats.com/fr/store/pizzeria-luigi-pepone-volontaires/smENXLR2RgOOratUkN3x0g'}, {'nom': 'PIZZA UNICA', 'url': 'https://www.ubereats.com/fr/store/pizza-unica/eA9XCbBsQSiW7IIC2L2vCw'}, {'nom': 'Sushi Crimée', 'url': 'https://www.ubereats.com/fr/store/sushi-crimee/B3w7No7KRBm8XOhRkxepzQ'}, {'nom': 'CHARBON ♨️ Maison des Grillades', 'url': 'https://www.ubereats.com/fr/store/charbon-maison-des-grillades/QjNsThz4VVe2-dNfhY4zmA'}, {'nom': 'Kyotorama', 'url': 'https://www.ubereats.com/fr/store/kyotorama-alfortville/SNd3slOOUjWgBGa2b0rFcg'}, {'nom': "Domino's Pizza", 'url': 'https://www.ubereats.com/fr/store/dominos-pizza-paris-2e/p6R3XK53QN-z3Q-LO6P2gg'}, {'nom': 'In Korea', 'url': 'https://www.ubereats.com/fr/store/in-korea/7SPTcZANVbC2MUgF6PWuhQ'}, {'nom': 'Le Cambodge', 'url': 'https://www.ubereats.com/fr/store/le-cambodge/A6mWcFVdRvyhOk04btP0cg'}, {'nom': 'Bolo Bolo 🥟 Cuisine de Hong Kong', 'url': 'https://www.ubereats.com/fr/store/bolo-bolo-cuisine-de-hong-kong/y_OKFnufWdKyhVaDxXvc1Q'}, {'nom': 'Sushi Shop', 'url': 'https://www.ubereats.com/fr/store/sushi-shop-reaumur/YDZ8iFZsTXel9-DsQFsUNA'}, {'nom': 'DimDimSum 🥟', 'url': 'https://www.ubereats.com/fr/store/dimdimsum/3U-povV-UxOnsGjCTS--QQ'}, {'nom': 'Nippon Shop', 'url': 'https://www.ubereats.com/fr/store/nippon-shop/Qy0cuf-BSuu08VlhIegrLg'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA'}, {'nom': 'Panda Panda', 'url': 'https://www.ubereats.com/fr/store/panda-panda/ah72XofTRaKP8d-sH-wimw'}, {'nom': 'PIZZA UNICA', 'url': 'https://www.ubereats.com/fr/store/pizza-unica/eA9XCbBsQSiW7IIC2L2vCw'}, {'nom': 'Sam Thai Audran', 'url': 'https://www.ubereats.com/fr/store/sam-thai-audran/3BUoHgbpS-6Va5DGOIJHHA'}, {'nom': 'Levantine', 'url': 'https://www.ubereats.com/fr/store/levantine/cM5-KvvjTT-m0HiLX99g1g'}, {'nom': 'Pizza Primavera 🍕', 'url': 'https://www.ubereats.com/fr/store/pizza-primavera-paris/HGHkqohYQ16XA7B2UhxuGA'}, {'nom': 'Pizzeria Luigi Pepone', 'url': 'https://www.ubereats.com/fr/store/pizzeria-luigi-pepone-volontaires/smENXLR2RgOOratUkN3x0g'}, {'nom': 'Neko Ramen 🍜', 'url': 'https://www.ubereats.com/fr/store/neko-ramen/GT8lBKdJT-mrf5U1WQgbgg'}, {'nom': 'Bises Nouilles 见面', 'url': 'https://www.ubereats.com/fr/store/bises-nouilles-见面/7OCSQTjmV5ifDGM7P_FBlw'}, {'nom': 'Trantranzai', 'url': 'https://www.ubereats.com/fr/store/trantranzai-chatelet-烫汤哉/KGWLbDl_Vn6_tgqLhYl_fQ'}, {'nom': 'MAMÉ kitchen', 'url': 'https://www.ubereats.com/fr/store/mame-kitchen-boetie/3KXUM5p8XiCTDs_KN-U_wQ'}, {'nom': 'Les Frères des Batignolles', 'url': 'https://www.ubereats.com/fr/store/les-freres-des-batignolles-paris-17/Obvfw642XSyu-KLur0akCw'}, {'nom': 'Hélia', 'url': 'https://www.ubereats.com/fr/store/helia/2hTzaCGCRgisqX0In-Qlug'}, {'nom': 'BARAK', 'url': 'https://www.ubereats.com/fr/store/barak-poissonniere/Ot_N-mDNQ3mUZPmjks9XQA'}, {'nom': 'La Taverne De Zhao', 'url': 'https://www.ubereats.com/fr/store/la-taverne-de-zhao-opera/DgJeFKdKXdyIV-QkjeHYzQ'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA'}, {'nom': 'Bambou Kitchen 🎋', 'url': 'https://www.ubereats.com/fr/store/bambou-kitchen/EdBObqJdWsOZLYjp7bMnxg'}, {'nom': 'La Boule Maître Kebabier', 'url': 'https://www.ubereats.com/fr/store/la-boule-maitre-kebabier-paris-2/8jfKJD_XTFeoUBf3ZeI1JA'}, {'nom': 'Street Bangkok Odéon', 'url': 'https://www.ubereats.com/fr/store/street-bangkok-odeon/g5aEKN46SX26a3zEnz_Yvg'}, {'nom': 'Banoi', 'url': 'https://www.ubereats.com/fr/store/banoi-vinaigriers/x5VW-aqqTgW2WPYxdVLvDg'}, {'nom': 'Crousty Game', 'url': 'https://www.ubereats.com/fr/store/crousty-game-paris-10/bn8vg5PjW7-BHuc4UEPdzg'}, {'nom': 'Bangcook Fast Good Thaï', 'url': 'https://www.ubereats.com/fr/store/bangcook-fast-good-thai-chateaudun/5hXFSrVuWyOhiEs34zTvGw'}, {'nom': 'SCHÖN Dürüm', 'url': 'https://www.ubereats.com/fr/store/schon-durum/7bNwYnE8REuRb4DmvTIvxQ'}, {'nom': 'Öz Maître Kebabier 🌯🇹🇷', 'url': 'https://www.ubereats.com/fr/store/oz-maitre-kebabier/ZE8BNzJAVz2MY9NQeluuRw'}, {'nom': 'PARSLEY Pita & Kebab 2ème', 'url': 'https://www.ubereats.com/fr/store/parsley-pita-%26-kebab-2eme/Z-wdTDSGWaGyUVfhM0u7Hw'}, {'nom': 'Les Nouilles Dansantes', 'url': 'https://www.ubereats.com/fr/store/les-nouilles-dansantes/KVWmnxgZQsa8xJoxE8k8iQ'}, {'nom': 'Il Grigio', 'url': 'https://www.ubereats.com/fr/store/il-grigio/nZNHxKukTUK26x_yatzpIg'}, {'nom': 'Fuchill 🇭🇰', 'url': 'https://www.ubereats.com/fr/store/fuchill/RADSimphXg2tuoyWnSxL8g?diningMode=DELIVERY'}, {'nom': 'Miuko Thé', 'url': 'https://www.ubereats.com/fr/store/miuko-the/vd30Yft7RyOQNNASkeb1mQ?diningMode=DELIVERY'}, {'nom': 'WANTED 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/wanted-smash-burgers-louvre/ufN0pyyiWXuog_wJpDevmA?diningMode=DELIVERY'}, {'nom': 'NOLA 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/nola-fried-chicken-louvre/5NoE6L5iXyGGFG6HiwdtlA?diningMode=DELIVERY'}, {'nom': 'OH MY CHEESE! 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/oh-my-cheese!-smash-burgers-louvre/WgBhgihUXZSHjXH5SbmgJg?diningMode=DELIVERY'}, {'nom': 'Gimmy', 'url': 'https://www.ubereats.com/fr/store/gimmy-louvre/zayTRlooUHKZ4AnsWFvhfw?diningMode=DELIVERY'}, {'nom': 'JUICY 🍔 Smash Burgers', 'url': 'https://www.ubereats.com/fr/store/juicy-smash-burgers-louvre/U79oQ_K2VHGcKJgo2FwyLA?diningMode=DELIVERY'}, {'nom': "S'MORE 🍪 American Desserts", 'url': 'https://www.ubereats.com/fr/store/smore-american-desserts-louvre/Lsyejam4VWmE7oAIiKfHTA?diningMode=DELIVERY'}, {'nom': 'Tacos & Fries', 'url': 'https://www.ubereats.com/fr/store/tacos-%26-fries-paris/0rinNWW7WHWp6Txhx5OAuw?diningMode=DELIVERY'}, {'nom': 'CIRCUS 🐔 Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/circus-fried-chicken-louvre/TMxVYItSXNmrBQBDZ9fg7w?diningMode=DELIVERY'}, {'nom': 'SEOULMATE 🐔 🇰🇷 Korean Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/seoulmate-korean-fried-chicken-louvre/9uffSqSdVC2e6QjE5y8viw?diningMode=DELIVERY'}, {'nom': 'Ça Croustille !', 'url': 'https://www.ubereats.com/fr/store/ca-croustille-!/WtF_NhuMX4iRIFXowOlCfw?diningMode=DELIVERY'}, {'nom': "Domino's Pizza", 'url': 'https://www.ubereats.com/fr/store/dominos-pizza-paris-2e/p6R3XK53QN-z3Q-LO6P2gg?diningMode=DELIVERY'}, {'nom': 'Blueberry Maki Bar', 'url': 'https://www.ubereats.com/fr/store/blueberry-maki-bar/_QE-aGENWWi62eg_WUVsWw?diningMode=DELIVERY'}, {'nom': "McDonald's®", 'url': 'https://www.ubereats.com/fr/store/mcdonalds-paris-louvre-rivoli/nWnUwvnuRBWFIRsBjSXlEQ?diningMode=DELIVERY'}, {'nom': 'Istanbul Charonne', 'url': 'https://www.ubereats.com/fr/store/istanbul-charonne/aqEOokA6Q46SPSEXkzOerw?diningMode=DELIVERY'}, {'nom': 'Isa Sushi', 'url': 'https://www.ubereats.com/fr/store/isa-sushi/nOhoCuXsVVmvC6IcfGJeow?diningMode=DELIVERY'}, {'nom': 'Kisoro Sushi', 'url': 'https://www.ubereats.com/fr/store/kisoro-sushi-louvre/KHH7i819UeaOAL3XrwaLBw?diningMode=DELIVERY'}, {'nom': 'Pizza Primavera 🍕', 'url': 'https://www.ubereats.com/fr/store/pizza-primavera-paris/HGHkqohYQ16XA7B2UhxuGA?diningMode=DELIVERY'}, {'nom': 'Osaki Sushi', 'url': 'https://www.ubereats.com/fr/store/osaki-sushi-bonne-nouvelle/R0cvvgd2XBydGGlgzrNxgg?diningMode=DELIVERY'}, {'nom': 'Ippudo Louvre', 'url': 'https://www.ubereats.com/fr/store/ippudo-louvre/DJ8bD_dsXlOPksDbIep-Jg?diningMode=DELIVERY'}, {'nom': 'Sushi Shop', 'url': 'https://www.ubereats.com/fr/store/sushi-shop-reaumur/YDZ8iFZsTXel9-DsQFsUNA?diningMode=DELIVERY'}, {'nom': 'In Korea', 'url': 'https://www.ubereats.com/fr/store/in-korea/7SPTcZANVbC2MUgF6PWuhQ?diningMode=DELIVERY'}, {'nom': 'Bolo Bolo 🥟 Cuisine de Hong Kong', 'url': 'https://www.ubereats.com/fr/store/bolo-bolo-cuisine-de-hong-kong/y_OKFnufWdKyhVaDxXvc1Q?diningMode=DELIVERY'}, {'nom': 'Neko Ramen 🍜', 'url': 'https://www.ubereats.com/fr/store/neko-ramen/GT8lBKdJT-mrf5U1WQgbgg?diningMode=DELIVERY'}, {'nom': 'Pizza Hut', 'url': 'https://www.ubereats.com/fr/store/pizza-hut-paris-9-cadet/PBYDjz7rT4OTMRfy3M0kFg?diningMode=DELIVERY'}, {'nom': 'Originals', 'url': 'https://www.ubereats.com/fr/store/originals-bobigny/Z6VcjluQWSmTMgACqW3bXg?diningMode=DELIVERY'}, {'nom': 'JOAYO', 'url': 'https://www.ubereats.com/fr/store/joayo-luxembourg/rjN2fOsqQaSyAv1z75l98g?diningMode=DELIVERY'}, {'nom': 'DimDimSum 🥟', 'url': 'https://www.ubereats.com/fr/store/dimdimsum/3U-povV-UxOnsGjCTS--QQ?diningMode=DELIVERY'}, {'nom': 'Le Cambodge', 'url': 'https://www.ubereats.com/fr/store/le-cambodge/A6mWcFVdRvyhOk04btP0cg?diningMode=DELIVERY'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA?diningMode=DELIVERY'}, {'nom': 'Benihana Paris', 'url': 'https://www.ubereats.com/fr/store/benihana-paris/gFZvbauYVbGsOlpSSfwPKA?diningMode=DELIVERY'}, {'nom': 'Suzie Wong by Panda Panda', 'url': 'https://www.ubereats.com/fr/store/suzie-wong-by-panda-panda/TyX-AdJ8WLagpa7__WyDhg?diningMode=DELIVERY'}, {'nom': 'Naruto Ramen Montmartre 🍜', 'url': 'https://www.ubereats.com/fr/store/naruto-ramen-montmartre/rP7ggoYcToOgq4nVp91PVg?diningMode=DELIVERY'}, {'nom': 'Pepe Chicken By FastGoodCuisine', 'url': 'https://www.ubereats.com/fr/store/pepe-chicken-by-fastgoodcuisine-grands-boulevards/jKwuK0BxVq-xg1jBBO4-tA?diningMode=DELIVERY'}, {'nom': 'Tiger Tiger by Panda Panda', 'url': 'https://www.ubereats.com/fr/store/tiger-tiger-by-panda-panda/NOTwhHXBTli5COm-FxbnzA?diningMode=DELIVERY'}, {'nom': 'Royal Kashmir', 'url': 'https://www.ubereats.com/fr/store/royal-kashmir/TgcHTC9cQVyZ5SwZeztvNQ?diningMode=DELIVERY'}, {'nom': 'Levantine', 'url': 'https://www.ubereats.com/fr/store/levantine/cM5-KvvjTT-m0HiLX99g1g?diningMode=DELIVERY'}, {'nom': 'BOLKIRI Street Food Viêt 🔥', 'url': 'https://www.ubereats.com/fr/store/bolkiri-street-food-viet-paris-11/pUF4HZiBUtm0Jt1L7UHwSw?diningMode=DELIVERY'}, {'nom': 'Don yoku', 'url': 'https://www.ubereats.com/fr/store/don-yoku/pQuhqpbLS8ONQ2K10syQQA?diningMode=DELIVERY'}, {'nom': 'Momidji', 'url': 'https://www.ubereats.com/fr/store/momidji/DDIiwB7HRgiS_X96vUN7Rw?diningMode=DELIVERY'}, {'nom': 'La Taverne De Zhao', 'url': 'https://www.ubereats.com/fr/store/la-taverne-de-zhao-opera/DgJeFKdKXdyIV-QkjeHYzQ?diningMode=DELIVERY'}, {'nom': 'PW restaurant coréen', 'url': 'https://www.ubereats.com/fr/store/pw-restaurant-coreen/ldE_1kUaSJypLbsfyPvpwg?diningMode=DELIVERY'}, {'nom': 'Pizzeria Luigi Pepone', 'url': 'https://www.ubereats.com/fr/store/pizzeria-luigi-pepone-volontaires/smENXLR2RgOOratUkN3x0g?diningMode=DELIVERY'}, {'nom': 'Capri Saint-Honoré', 'url': 'https://www.ubereats.com/fr/store/capri-saint-honore/kUKEn1dbQqSnLNU-FJMDTQ?diningMode=DELIVERY'}, {'nom': 'DongNe Chicken', 'url': 'https://www.ubereats.com/fr/store/dongne-chicken/KQtl8gnrXYi1eG_cSl7GEQ?diningMode=DELIVERY'}, {'nom': 'Trantranzai', 'url': 'https://www.ubereats.com/fr/store/trantranzai-chatelet-烫汤哉/KGWLbDl_Vn6_tgqLhYl_fQ?diningMode=DELIVERY'}, {'nom': 'Street Bangkok Odéon', 'url': 'https://www.ubereats.com/fr/store/street-bangkok-odeon/g5aEKN46SX26a3zEnz_Yvg?diningMode=DELIVERY'}, {'nom': '대장 DAEJANG', 'url': 'https://www.ubereats.com/fr/store/대장-daejang/n12Fmn-7WEeZr268Yd3_2A?diningMode=DELIVERY'}, {'nom': 'Matsuri', 'url': 'https://www.ubereats.com/fr/store/matsuri-rue-du-bac/friCHm7fQfGfcMZlirE24w?diningMode=DELIVERY'}, {'nom': "O'Tacos", 'url': 'https://www.ubereats.com/fr/store/otacos-chatelet/iI439XtiRfSffGqy-aUJRw?diningMode=DELIVERY'}, {'nom': 'Le Mabillon Brasserie Parisienne', 'url': 'https://www.ubereats.com/fr/store/le-mabillon-brasserie-parisienne/uUjop19fWj6RX2fnGLCjiQ?diningMode=DELIVERY'}, {'nom': 'Jeju 제주시 restaurant coréen 🇰🇷', 'url': 'https://www.ubereats.com/fr/store/jeju-제주시-restaurant-coreen/35trtsWFUwaI1wgFJgEAKQ?diningMode=DELIVERY'}, {'nom': 'Trois Fois Plus de Piment', 'url': 'https://www.ubereats.com/fr/store/trois-fois-plus-de-piment-saint-martin/nzcLzXHnRlKl2nCobsfpxw?diningMode=DELIVERY'}, {'nom': 'Carrefour', 'url': 'https://www.ubereats.com/fr/store/carrefour-paris-richelieu-43/esDV0no9TuqkdkvAnpwCUg?diningMode=DELIVERY'}, {'nom': 'Sam Thai Audran', 'url': 'https://www.ubereats.com/fr/store/sam-thai-audran/3BUoHgbpS-6Va5DGOIJHHA?diningMode=DELIVERY'}, {'nom': 'Pan Asie', 'url': 'https://www.ubereats.com/fr/store/pan-asie/2_w0YIR7VPm5pHGblFG7hg?diningMode=DELIVERY'}, {'nom': 'Nippon Shop', 'url': 'https://www.ubereats.com/fr/store/nippon-shop/Qy0cuf-BSuu08VlhIegrLg?diningMode=DELIVERY'}, {'nom': 'SHIFU', 'url': 'https://www.ubereats.com/fr/store/shifu/rHvM9AyJQqG52wFQvXfWag?diningMode=DELIVERY'}, {'nom': "Carin'o Pizza", 'url': 'https://www.ubereats.com/fr/store/carino-pizza/hRRwgECSRq-kNg6iPAAZhg?diningMode=DELIVERY'}, {'nom': 'Monoprix', 'url': 'https://www.ubereats.com/fr/store/monoprix-sebastopol/w-DiQyM2XWGtS-L85E5KVA?diningMode=DELIVERY'}, {'nom': 'Casa di Peppe by Peppe Pizzeria', 'url': 'https://www.ubereats.com/fr/store/casa-di-peppe-by-peppe-pizzeria/sn6c4neJXamEuhzDsoHnFA?diningMode=DELIVERY'}, {'nom': 'Bouillon Service ! République', 'url': 'https://www.ubereats.com/fr/store/bouillon-service-!-republique/P5giWaDmVnipOkjtiyMjDg?diningMode=DELIVERY'}, {'nom': 'Kisoro', 'url': 'https://www.ubereats.com/fr/store/kisoro-montmartre/XFb_QPTCVqmZWzVhwiKyYA?diningMode=DELIVERY'}, {'nom': 'Napoli Gang by Big Mamma', 'url': 'https://www.ubereats.com/fr/store/napoli-gang-by-big-mamma-batignolles/s3b64uWaQ2iBX0OeebKX0A?diningMode=DELIVERY'}, {'nom': 'Dandino Restaurant Italien', 'url': 'https://www.ubereats.com/fr/store/dandino-restaurant-italien/waKI1deoVkOzYYhBDAQm6g?diningMode=DELIVERY'}, {'nom': "L'As du Shawarma", 'url': 'https://www.ubereats.com/fr/store/las-du-shawarma-paris-12/lS71zxmIU2WmsSELK9kDTw?diningMode=DELIVERY'}, {'nom': 'Chipotle Mexican Grill', 'url': 'https://www.ubereats.com/fr/store/chipotle-mexican-grill-montmartre/mR9vNHQNTa6RY2P5tKOjcQ?diningMode=DELIVERY'}, {'nom': 'Cô My Cantine', 'url': 'https://www.ubereats.com/fr/store/co-my-cantine/Kachu8IcTten2cwgR_sJiw?diningMode=DELIVERY'}, {'nom': 'SHABESTAN', 'url': 'https://www.ubereats.com/fr/store/shabestan-paris-8/-2feaZGhQdeH3Qnd9Cyi6A?diningMode=DELIVERY'}, {'nom': 'Pokawa', 'url': 'https://www.ubereats.com/fr/store/pokawa-chateaudun/bfBN-02RQ36iNsdr4SNfow?diningMode=DELIVERY'}, {'nom': 'OKAYA Sushi', 'url': 'https://www.ubereats.com/fr/store/okaya-sushi/aCFK7cCLXmqz0TIQguXc6Q?diningMode=DELIVERY'}, {'nom': 'Blend Hamburger', 'url': 'https://www.ubereats.com/fr/store/blend-hamburger-madeleine/cey-XJ0MTfyJYAKfNMV-fA?diningMode=DELIVERY'}, {'nom': 'Asia Room 川香阁', 'url': 'https://www.ubereats.com/fr/store/asia-room-川香阁/UdP00XrLTA61jnrTQRSrWw?diningMode=DELIVERY'}, {'nom': 'African Evasion', 'url': 'https://www.ubereats.com/fr/store/african-evasion-ivry/RHfVmqOLXT6J6QzWhKwfzQ?diningMode=DELIVERY'}, {'nom': 'Panda Panda', 'url': 'https://www.ubereats.com/fr/store/panda-panda/ah72XofTRaKP8d-sH-wimw?diningMode=DELIVERY'}, {'nom': 'Intermarché', 'url': 'https://www.ubereats.com/fr/store/intermarche-quais-de-seine/t4-U3F9YUqeqzYoEu0YO4w?diningMode=DELIVERY'}, {'nom': 'Yatai Ramen', 'url': 'https://www.ubereats.com/fr/store/yatai-ramen-pyramide/DPIAIhoIV3-S53YlJ1ZudQ?diningMode=DELIVERY'}, {'nom': 'Bangcook Fast Good Thaï', 'url': 'https://www.ubereats.com/fr/store/bangcook-fast-good-thai-chateaudun/5hXFSrVuWyOhiEs34zTvGw?diningMode=DELIVERY'}, {'nom': 'Côté Sushi', 'url': 'https://www.ubereats.com/fr/store/cote-sushi-rambuteau/WgHoWLCRSTCEtfKOPMd_wA?diningMode=DELIVERY'}, {'nom': 'Homer Lobster 🦞', 'url': 'https://www.ubereats.com/fr/store/homer-lobster-odeon/iGu9uMFJXU68zj04YokaHQ?diningMode=DELIVERY'}, {'nom': 'Popeyes Louisiana Chicken', 'url': 'https://www.ubereats.com/fr/store/popeyes-louisiana-chicken-republique/cgKyTjTaVOyN_X76LmgKfA?diningMode=DELIVERY'}, {'nom': 'Krousty Sabaïdi', 'url': 'https://www.ubereats.com/fr/store/krousty-sabaidi-chatelet/2_5Q6WfmUKKb523t2wXP4A?diningMode=DELIVERY'}, {'nom': 'Le Camion Qui Fume', 'url': 'https://www.ubereats.com/fr/store/le-camion-qui-fume-odeon/TPMEm6PyUtORJICJfxxJHA?diningMode=DELIVERY'}, {'nom': 'Kapunka Cantine Thaï', 'url': 'https://www.ubereats.com/fr/store/kapunka-cantine-thai-montorgueil/79HsuLFAT3OmQqHOi7BF5Q?diningMode=DELIVERY'}, {'nom': 'Maison de Chengdu 古味成都', 'url': 'https://www.ubereats.com/fr/store/maison-de-chengdu-古味成都/2YHEVxiSWru79NEcJMNr2w?diningMode=DELIVERY'}, {'nom': 'K-Chimaek 🇰🇷 Goncourt', 'url': 'https://www.ubereats.com/fr/store/k-chimaek-goncourt/2iYkPfOFUN6lZahEAer6QA?diningMode=DELIVERY'}, {'nom': 'Basilic Thaï', 'url': 'https://www.ubereats.com/fr/store/basilic-thai/I97d0LYgREe-5BzhCGapwA?diningMode=DELIVERY'}, {'nom': 'Big Fernand', 'url': 'https://www.ubereats.com/fr/store/big-fernand-saint-honore/_vgdLZuFTXa1zxYQ-YMKLQ?diningMode=DELIVERY'}, {'nom': 'Le Royal China', 'url': 'https://www.ubereats.com/fr/store/le-royal-china/LNjFMEZBSmK2VERqaufIdg?diningMode=DELIVERY'}, {'nom': 'Ete Edo', 'url': 'https://www.ubereats.com/fr/store/ete-edo-11e/dkGBpeW8R-aBiRLG2rSy7w?diningMode=DELIVERY'}, {'nom': 'LÜKS Kebab', 'url': 'https://www.ubereats.com/fr/store/luks-kebab-poissonniere/nqfcKbHAX7S6aY8xdVA6ZQ?diningMode=DELIVERY'}, {'nom': 'My Noodles', 'url': 'https://www.ubereats.com/fr/store/my-noodles/cn6vxGcBQaaYlgI8hBySMw?diningMode=DELIVERY'}, {'nom': 'Taj Mahal', 'url': 'https://www.ubereats.com/fr/store/taj-mahal-simon-le-franc/62SKZ65VT96Qv0vVyftP4w?diningMode=DELIVERY'}, {'nom': 'Hoki Sushi', 'url': 'https://www.ubereats.com/fr/store/hoki-sushi/4nyuop-jSeWtzgWodbTAew?diningMode=DELIVERY'}, {'nom': 'Kare Ramen & Curry', 'url': 'https://www.ubereats.com/fr/store/kare-ramen-%26-curry/3SVnBL_fXUWwuaXN73EtWg?diningMode=DELIVERY'}, {'nom': "Saveurs d'Asie Maubert", 'url': 'https://www.ubereats.com/fr/store/saveurs-dasie-maubert/-0Ot4DcUToCOiJOfdvV-Vg?diningMode=DELIVERY'}, {'nom': 'Bocamexa Odéon', 'url': 'https://www.ubereats.com/fr/store/bocamexa-odeon-restaurant-mexicain/BvYoFGbVT92QcqzADx0DLg'}, {'nom': 'Coffee Parisien Princesse', 'url': 'https://www.ubereats.com/fr/store/coffee-parisien-princesse/syew2Si7Vc-qgLREWz6zhA'}, {'nom': 'Pitaya', 'url': 'https://www.ubereats.com/fr/store/pitaya-la-harpe/_g1P5l77XP6rAepXylxgkA'}, {'nom': 'INDIAN MAAN MAHAL RESTAURANT', 'url': 'https://www.ubereats.com/fr/store/indian-maan-mahal-restaurant/-S2OZB3kSvCVbyQKmKkvkQ'}, {'nom': 'Tripletta', 'url': 'https://www.ubereats.com/fr/store/tripletta-gaite/43LaMF1PVPO14c-Bg3DpEg'}, {'nom': 'A Folé Paris 19', 'url': 'https://www.ubereats.com/fr/store/a-fole-paris-19/ELuUypBmSf2VKPrG_PACRA'}, {'nom': 'Marcus Pizze Amore', 'url': 'https://www.ubereats.com/fr/store/marcus-pizze-amore/AqPClxLeQDCIpVjYoBlEJw'}, {'nom': 'Five Pizza Original', 'url': 'https://www.ubereats.com/fr/store/five-pizza-original-reaumur/X2qlynhRT3K0EyqUouT36A'}, {'nom': 'Tasty Nouilles', 'url': 'https://www.ubereats.com/fr/store/tasty-nouilles/h6YJqrCeTimj51chVbxR9w'}, {'nom': 'GRUPPOMIMO 🍕🍝\u200e \u200e\u200e', 'url': 'https://www.ubereats.com/fr/store/gruppomimo-\u200e-\u200e\u200e-11eme/0ohz2-BsWnSqnl1UnUMVgQ'}, {'nom': 'Sakura Paris', 'url': 'https://www.ubereats.com/fr/store/sakura-paris/u9oRwt4jRrmHeewk4I5K-g'}, {'nom': 'Nossa', 'url': 'https://www.ubereats.com/fr/store/nossa-paris-5eme/qvrMxvWyQf2poWNITJKabg'}, {'nom': 'Nobi Nobi', 'url': 'https://www.ubereats.com/fr/store/nobi-nobi-paris/GKakxI_dSPyg5LTAEnF0ug'}, {'nom': 'Okinawa', 'url': 'https://www.ubereats.com/fr/store/okinawa-gentilly/FvkYNFH4RP2a_Xcz8zur5g'}, {'nom': 'Quick', 'url': 'https://www.ubereats.com/fr/store/quick-sebastopol/j0JTAGqUS9C6er5x8ZdcZw'}, {'nom': 'Okito', 'url': 'https://www.ubereats.com/fr/store/okito-montrouge/vW7bvrbWTUKu_Su8wa3yzw'}, {'nom': 'Jinchan Yokocho', 'url': 'https://www.ubereats.com/fr/store/jinchan-yokocho/4w1AvqKAVMeGRrkdXqvPPQ'}, {'nom': 'Pizza Cosy', 'url': 'https://www.ubereats.com/fr/store/pizza-cosy-paris-5/4sRtxOteTjWPpvo3s_FazA'}, {'nom': 'Kesmë', 'url': 'https://www.ubereats.com/fr/store/kesme-val-de-grace/dxcduCwsU6urfdqPKYfNGA'}, {'nom': 'Woodiz Pizza au feu de bois', 'url': 'https://www.ubereats.com/fr/store/woodiz-pizza-au-feu-de-bois-paris-14/uf46z55rQOGQc1RXgIYC6A'}, {'nom': 'LA BROCHE Kebab Berlinois 🥙', 'url': 'https://www.ubereats.com/fr/store/la-broche-kebab-berlinois-paris-5/vxWgFI7AVRC3Z0xUwXsKIg'}, {'nom': 'Starsmash by Amixem', 'url': 'https://www.ubereats.com/fr/store/starsmash-by-amixem-grands-boulevards/U8M0o8bZUaKleV-G1rIoIw'}, {'nom': 'Superbao', 'url': 'https://www.ubereats.com/fr/store/superbao/1t_Bx39cX4eIjj1KBG-ouw'}, {'nom': 'Los Mexicanos 🌮', 'url': 'https://www.ubereats.com/fr/store/los-mexicanos/RcqblpADRbiQJ-gRhmKHRA'}, {'nom': 'Burger & Fries  Odéon', 'url': 'https://www.ubereats.com/fr/store/burger-%26-fries-odeon/XWE_S-pvS6mBNLipM3bNyg'}, {'nom': 'La Boule Maître Kebabier', 'url': 'https://www.ubereats.com/fr/store/la-boule-maitre-kebabier-paris-2/8jfKJD_XTFeoUBf3ZeI1JA'}, {'nom': 'Peppe Pizzeria', 'url': 'https://www.ubereats.com/fr/store/peppe-pizzeria-etienne-marcel/Li1LauY4UYaKdvr00c2Y6A'}, {'nom': 'Chô Chaï', 'url': 'https://www.ubereats.com/fr/store/cho-chai-paris/VEbnxBRRVSessMXnX0jgcw'}, {'nom': 'Dumbo Marais', 'url': 'https://www.ubereats.com/fr/store/dumbo-marais/G76ZprxUWJ2vwD-4W4qwYA'}, {'nom': 'Jinchan Shokudo', 'url': 'https://www.ubereats.com/fr/store/jinchan-shokudo/1KZFURJVTJefOs2dreP4kg'}, {'nom': 'BMK Folie-Bamako', 'url': 'https://www.ubereats.com/fr/store/bmk-folie-bamako/zQ6dAjkAT7mduXwj3WsHeg'}, {'nom': 'Pizzou', 'url': 'https://www.ubereats.com/fr/store/pizzou-marais/EKh4k9x4WnixfYknNUaOSg'}, {'nom': 'Poulet Addict', 'url': 'https://www.ubereats.com/fr/store/poulet-addict-rue-monge/oXB9SUA1U7epm6-6hNrE6w'}, {'nom': 'Pokeshop Saint Michel', 'url': 'https://www.ubereats.com/fr/store/pokeshop-saint-michel/5PV1IUZsWI21mlpMX2tlog'}, {'nom': 'NooRI 🌮🍣', 'url': 'https://www.ubereats.com/fr/store/noori/HZH9eSf9XC2FaaxjRC1U2A'}, {'nom': 'THE SLICERIE', 'url': 'https://www.ubereats.com/fr/store/the-slicerie/d82IoODRVXGHHMQ40Zv3NA'}, {'nom': 'PB Poulet Braisé', 'url': 'https://www.ubereats.com/fr/store/pb-poulet-braise-chatelet/lCbsqGCBUDKYHoBI0vXrEg'}, {'nom': 'Avec Amour Le Burger', 'url': 'https://www.ubereats.com/fr/store/avec-amour-le-burger-montparnasse/pg3W8jyFWKGWel6PK6PoFg'}, {'nom': 'Khao Thai', 'url': 'https://www.ubereats.com/fr/store/khao-thai-restaurant-thailandais/i6IL1jcORJyEJfkrF-uRDw'}, {'nom': '21G Dumpling', 'url': 'https://www.ubereats.com/fr/store/21g-dumpling/7mxPSf33Rhu1SmOv-Ko9OA'}, {'nom': 'Hokkaido 8', 'url': 'https://www.ubereats.com/fr/store/hokkaido-8/NKlVb8WYSQSbunHr9FrY9g'}, {'nom': 'Carrefour Sprint', 'url': 'https://www.ubereats.com/fr/store/carrefour-sprint-paris-jussieu-35/6eL98dQMX5WbqGa1rRNGKA'}, {'nom': 'Kawok®', 'url': 'https://www.ubereats.com/fr/store/kawok-chatillon/ygo-H-_BUIW-jUizHhKKFw'}, {'nom': 'Dalmata', 'url': 'https://www.ubereats.com/fr/store/dalmata/3ONjk4tvS6O5cy5f3NhZNw'}, {'nom': 'PNY GAITÉ', 'url': 'https://www.ubereats.com/fr/store/pny-gaite/9tgP0QMhU3a4r9M-Dvgusw'}, {'nom': 'ISSHIN Ramen Olympiades 🇯🇵', 'url': 'https://www.ubereats.com/fr/store/isshin-ramen-olympiades/2TDOGrohVkiUPklYw2giIw'}, {'nom': 'LABEL’GE FRITE  Le roi d’la frite 🇧🇪', 'url': 'https://www.ubereats.com/fr/store/labelge-frite-le-roi-dla-frite/QlnzbvgOT0ar8QUVHA93Nw'}, {'nom': 'SUSHI ALESIA', 'url': 'https://www.ubereats.com/fr/store/sushi-alesia/-FgKLnMmUuKIiQtvRInSSQ'}, {'nom': 'India Gate', 'url': 'https://www.ubereats.com/fr/store/india-gate/Ne7BpODOQfqqxHwSESYUqg'}, {'nom': 'Round Egg Buns', 'url': 'https://www.ubereats.com/fr/store/round-egg-buns-paris-2/RPjvSs12VQS3YzrAV5DSEw'}, {'nom': 'THUY Street Food Viêt République', 'url': 'https://www.ubereats.com/fr/store/thuy-street-food-viet-republique/PYz4Zkv4WNi-XMU3zyUOJA'}, {'nom': 'BASIS Grilled Kebab', 'url': 'https://www.ubereats.com/fr/store/basis-grilled-kebab-belleville/olLcGnUvUs-ZUzrDLdu4Bg'}, {'nom': "La Grillade Parisienne Aux Saveurs D'Afrique", 'url': 'https://www.ubereats.com/fr/store/la-grillade-parisienne-aux-saveurs-dafrique/Q6_XD_vgQxmYy49qDJh-iQ'}, {'nom': '🇰🇷 OLIVE CHICKEN', 'url': 'https://www.ubereats.com/fr/store/olive-chicken-korean-fried-chicken/ky2g6ga-R9-dfd1AifX2qw'}, {'nom': "Starvin' Joe", 'url': 'https://www.ubereats.com/fr/store/starvin-joe-burgers/hwgn_CBbQnicAhVV31LLxg'}, {'nom': 'PNY  MARAIS TEMPLE', 'url': 'https://www.ubereats.com/fr/store/pny-marais-temple/Jlm1RHJoXme41yERQIlTJQ'}, {'nom': 'Afri Call Food Paris Gare De Lyon', 'url': 'https://www.ubereats.com/fr/store/afri-call-food-paris-gare-de-lyon/MnlBL2GBSByLfvRtmwEHrA'}, {'nom': 'Père & Fish', 'url': 'https://www.ubereats.com/fr/store/pere-%26-fish-paris/b7mrNmgsRU2L-SUnqWwzrw'}, {'nom': 'Chamas Tacos', 'url': 'https://www.ubereats.com/fr/store/chamas-tacos-chatelet-les-halles/-mJhKryAT7y08IayplVAYw'}, {'nom': 'One and One Burger', 'url': 'https://www.ubereats.com/fr/store/one-and-one-burger/RMs4jUKRXlGwMpDwkFUXiQ'}, {'nom': 'Naamaste India', 'url': 'https://www.ubereats.com/fr/store/naamaste-india/cb60bYEFQaymDTDrzlKnXw'}, {'nom': 'Enooba Tunisian Street Food', 'url': 'https://www.ubereats.com/fr/store/enooba-tunisian-street-food/o7w4DSOzSzOkz17_xwIa9Q'}, {'nom': 'Izu sushi', 'url': 'https://www.ubereats.com/fr/store/izu-sushi/5rgQ5KrwRkmU0hvmcnL3Yg'}, {'nom': 'Shiso Burger', 'url': 'https://www.ubereats.com/fr/store/shiso-burger/z09F9qSDQR2EcoF73WljjA'}, {'nom': 'Asian Food by BAZE', 'url': 'https://www.ubereats.com/fr/store/asian-food-by-baze-ivry/VBjq_WV1UJOg1MiqGA-_lg'}, {'nom': 'Joe & The Juice', 'url': 'https://www.ubereats.com/fr/store/joe-%26-the-juice-victor-hugo/rlhbunnfRRa0-V4Zk-UU8A'}, {'nom': 'Clasico Argentino Empanadas 🇦🇷  Madame', 'url': 'https://www.ubereats.com/fr/store/clasico-argentino-empanadas-madame/Ae3AEFFDRgmB_sNoZv_zWw'}, {'nom': 'Nagoya', 'url': 'https://www.ubereats.com/fr/store/nagoya/bnDLldE_TqagpoJZE7sxTw'}, {'nom': "Winner's chicken", 'url': 'https://www.ubereats.com/fr/store/winners-chicken/0Yhvh2tXQVKRejRqMS5Qig'}, {'nom': "L'Atelier Artisan Crêpier", 'url': 'https://www.ubereats.com/fr/store/latelier-artisan-crepier-mabillon/POzjVdaWW5W7QJAFkrj1BA'}, {'nom': 'Kuma Curry et Donburi', 'url': 'https://www.ubereats.com/fr/store/kuma-curry-et-donburi/RGkbMRxsRB-yZ08AeLl8lA'}, {'nom': 'Nach!', 'url': 'https://www.ubereats.com/fr/store/nach!-paris-17/_tD_MfveXlWATsFrwEqGxw'}, {'nom': 'GANGS', 'url': 'https://www.ubereats.com/fr/store/gangs-bastille/QaJMo7gHXdSIbZoXkInrNg'}, {'nom': 'Spécimen Burger', 'url': 'https://www.ubereats.com/fr/store/specimen-burger/BBLZcspLX_G58oFqwgkTzw'}, {'nom': 'TANORI Arts et Métiers', 'url': 'https://www.ubereats.com/fr/store/tanori-arts-et-metiers/b35EWP86V-aP76yCdZP9pQ'}, {'nom': 'Pho Kitchen', 'url': 'https://www.ubereats.com/fr/store/pho-kitchen/TdsNYXe5Qh6lbh7KSSfz6w'}, {'nom': 'Taj Mahal Paris', 'url': 'https://www.ubereats.com/fr/store/taj-mahal-paris/ZCKe4BIGRCeicmmhpWqhXQ'}, {'nom': 'Kalki', 'url': 'https://www.ubereats.com/fr/store/kalki-restaurant-traditionnel-indien/x_cnF7yfQKqjiiKJR_UAMg'}, {'nom': 'Pure Veg : Indian Cuisine', 'url': 'https://www.ubereats.com/fr/store/pure-veg-%3A-indian-cuisine/YGELP_giRG-CdoPbt7uSfw'}, {'nom': "La Bel'ge frite 🇧🇪 Saint Michel", 'url': 'https://www.ubereats.com/fr/store/la-belge-frite-saint-michel/aOPu0tYjWQmEfcoG3xwl7Q'}, {'nom': 'MJ Sushi', 'url': 'https://www.ubereats.com/fr/store/mj-sushi/py0TDOEFQaqg7hILnTcXAA'}, {'nom': 'IT Italian Trattoria', 'url': 'https://www.ubereats.com/fr/store/it-italian-trattoria-bnf/ysYqPoFlTa-EhTP8jBJWRQ'}, {'nom': 'Los Pollos', 'url': 'https://www.ubereats.com/fr/store/los-pollos-bastille/yZBX56nWXPuYkz3iEmU8Aw'}, {'nom': 'G La Dalle', 'url': 'https://www.ubereats.com/fr/store/g-la-dalle-chatelet/zlW2cff4TJiceyAQI0c2Vg'}, {'nom': 'Franprix', 'url': 'https://www.ubereats.com/fr/store/franprix-paris-antoine/cpXQy7pvSiupzS0he6wlgQ'}, {'nom': 'Makham thaï', 'url': 'https://www.ubereats.com/fr/store/makham-thai/xfszwX8nQZiBrF60lVxEbg'}, {'nom': 'Piou!Piou!', 'url': 'https://www.ubereats.com/fr/store/piou!piou!-saint-denis/Y64ee8eqVN-Lgn_bdHDa9w'}, {'nom': 'Meister Berliner Kebap', 'url': 'https://www.ubereats.com/fr/store/meister-berliner-kebap-montparnasse/iaNlE5JOSlSCUI5S1Gi4DQ'}, {'nom': 'Super U', 'url': 'https://www.ubereats.com/fr/store/super-u-paris-paul-bert/-SoZqiULSx-s0zsOr8oyzg'}, {'nom': 'Riz Djondjon', 'url': 'https://www.ubereats.com/fr/store/riz-djondjon/_AsExHIZR-GT3V-FWyfT3g'}, {'nom': 'Sushiko', 'url': 'https://www.ubereats.com/fr/store/sushiko/maAdrggeQTyD2Cs3r4P1PA'}, {'nom': 'La Belle Vie', 'url': 'https://www.ubereats.com/fr/store/la-belle-vie-paris/3EJWfVsRWx-dEyF6xm45dA'}, {'nom': '5PIZZ 🍕', 'url': 'https://www.ubereats.com/fr/store/5pizz-paris-12/VRDrreO6TNSMG2FLkhXOJg'}, {'nom': 'Mao Fry', 'url': 'https://www.ubereats.com/fr/store/mao-fry/0_gd9mkwWdOhVh26SN4n8w'}, {'nom': 'Little Apple', 'url': 'https://www.ubereats.com/fr/store/little-apple/sQWB_FI3STGUzg5oBxqf-g'}, {'nom': 'Happy Nouilles', 'url': 'https://www.ubereats.com/fr/store/happy-nouilles/rPjM-XzgSZazYTldnxWZSQ'}, {'nom': 'Garba Chaud', 'url': 'https://www.ubereats.com/fr/store/garba-chaud-vitry/5TYuILSFUpCDds7TdMcGlA'}, {'nom': "Sweetea's restaurant coréen 🇰🇷 Tolbiac", 'url': 'https://www.ubereats.com/fr/store/sweeteas-restaurant-coreen-tolbiac/Qdx05g0fX5-uTW7KEyceJg'}, {'nom': 'Pho 13', 'url': 'https://www.ubereats.com/fr/store/pho-13/FUSCxynhTo-_oW3PRHRr1w'}, {'nom': 'Enjoy Tacos®', 'url': 'https://www.ubereats.com/fr/store/enjoy-tacos-paris-15/L10TF--HXGCnQ9-yuA5hzQ'}, {'nom': 'Dumbo Oberkampf', 'url': 'https://www.ubereats.com/fr/store/dumbo-oberkampf/sonWtkA-V6SvELrBJ6Y6kg'}, {'nom': 'Sushi Oh', 'url': 'https://www.ubereats.com/fr/store/sushi-oh/X2zU8iFYQ7KQY3FQkTS2BQ'}, {'nom': 'Malak Al Tawouk', 'url': 'https://www.ubereats.com/fr/store/malak-al-tawouk-jussieu/CNmjvZZYUOG-6O1BaiJkaA'}, {'nom': 'Muki Sushi', 'url': 'https://www.ubereats.com/fr/store/muki-sushi/Fz7gX2E-TEyDN4WIvqBC9Q'}, {'nom': 'La Vie Dirty Vegan Burgers', 'url': 'https://www.ubereats.com/fr/store/la-vie-dirty-vegan-burgers-grands-boulevards/pDKfUMGMXLKVXaKBGhYFjQ'}, {'nom': 'Il était un burger 🍔', 'url': 'https://www.ubereats.com/fr/store/il-etait-un-burger/_6XYsKT7UZObF_qXMATaPQ'}, {'nom': "Shawar'Mama", 'url': 'https://www.ubereats.com/fr/store/shawarmama-paris-11/oDI9ls2HWWiHSkXzD9fyNw'}, {'nom': 'FAT PIZZA 🍕 Vanves', 'url': 'https://www.ubereats.com/fr/store/fat-pizza-vanves/lN_dyxtAUXyL6wtrsxoUUQ'}, {'nom': 'Sangeet', 'url': 'https://www.ubereats.com/fr/store/sangeet/5iwcRcs1RZuX2SfjaMWPAw'}, {'nom': 'Theory Vegan Burgers & Bowls', 'url': 'https://www.ubereats.com/fr/store/theory-vegan-burgers-%26-bowls-paris/9l6aNsmUT7iSAu9rO8sRCg'}, {'nom': 'GRUPPOMIMO 🍕🍝', 'url': 'https://www.ubereats.com/fr/store/gruppomimo-vincennes/Zqw6orc9VlGnAvPtwTJ-jw'}, {'nom': 'F**KING SMASH by Avec Amour', 'url': 'https://www.ubereats.com/fr/store/f**king-smash-by-avec-amour-montparnasse/Z-4IN9PYViKnpMdJUKFhjw'}, {'nom': 'ITAKA SUSHI', 'url': 'https://www.ubereats.com/fr/store/itaka-sushi/qDpV_RWBXaCrUDRGyN6QOg'}, {'nom': 'ZAMANE', 'url': 'https://www.ubereats.com/fr/store/zamane/_5Q3pMEISWCy6h9rwJlnxg'}, {'nom': 'Viet Nam Impressions', 'url': 'https://www.ubereats.com/fr/store/viet-nam-impressions/h1gdxCHXS2CR2w5pvVZQ5g'}, {'nom': 'Amore e Gusto', 'url': 'https://www.ubereats.com/fr/store/amore-e-gusto-paris-5eme/6kxVxqaTTNSQUik91kNWzA'}, {'nom': '🌮 Tacos & Burger Delice 🍔', 'url': 'https://www.ubereats.com/fr/store/tacos-%26-burger-delice/GG5mDAD9V_qzdWYvRNp-ZA'}, {'nom': 'Vincent Salur', 'url': 'https://www.ubereats.com/fr/store/vincent-salur-15eme/rNTpeBMNW-y7JzVeXbNHMw'}, {'nom': 'Crousty One', 'url': 'https://www.ubereats.com/fr/store/crousty-one-oberkampf/X7jibuuVSYexUgXq0rKohA'}, {'nom': 'Yaai Thaï', 'url': 'https://www.ubereats.com/fr/store/yaai-thai-daguerre/VUVvMgTFSxKByQRHmM_VSg'}, {'nom': 'Planet Tieb', 'url': 'https://www.ubereats.com/fr/store/planet-tieb/lVUFHBCISMCq4fhOlm0bxQ'}, {'nom': 'Clasico Argentino Empanadas 🇦🇷 Cotte', 'url': 'https://www.ubereats.com/fr/store/clasico-argentino-empanadas-cotte/Y7T62J4VRw-GYS9iQiRuVw'}, {'nom': 'Le Côté Sucré🍭', 'url': 'https://www.ubereats.com/fr/store/le-cote-sucre-paris/EzPYis3dQJmsT38-gtUN-Q'}, {'nom': 'Rollsy', 'url': 'https://www.ubereats.com/fr/store/rollsy/GI_KiPLrVHGVRHmdeB3gxw'}, {'nom': 'Clasico Argentino Empanadas 🇦🇷  Marais', 'url': 'https://www.ubereats.com/fr/store/clasico-argentino-empanadas-marais/J6YrSBqMSIexGDbnAuDsKA'}, {'nom': 'Soho Smash Burger', 'url': 'https://www.ubereats.com/fr/store/soho-smash-burger/08sgd0lHTU-35aUfSEV2yg'}, {'nom': 'Le Why Not', 'url': 'https://www.ubereats.com/fr/store/le-why-not/AXTRT5EtTbSXrFf4Jnul5A'}, {'nom': "L'original Riz Crousti", 'url': 'https://www.ubereats.com/fr/store/loriginal-riz-crousti-ivry/9JxrvUu9VlSf2Ed8-XXhFQ'}, {'nom': 'Biryani Palace', 'url': 'https://www.ubereats.com/fr/store/biryani-palace/FA8wLA-rUXSjSZrb8lmXRg'}, {'nom': 'Tantie J’ai Faim', 'url': 'https://www.ubereats.com/fr/store/tantie-jai-faim/pp5caOZgS-qVV3-ek7FIxg'}, {'nom': 'Le Constantinople Les 3 Broches 🥙', 'url': 'https://www.ubereats.com/fr/store/le-constantinople-les-3-broches/4GU5etZaRPO0bmG_-YGRZQ'}, {'nom': 'Hêvî Kebab', 'url': 'https://www.ubereats.com/fr/store/hevi-kebab/7CFdcpN-RWi9d8g2aY8uzg'}, {'nom': 'Tsuru', 'url': 'https://www.ubereats.com/fr/store/tsuru/y3IufvL0VOiZCoNoQiqNmA'}, {'nom': "CHIV'S KOY", 'url': 'https://www.ubereats.com/fr/store/chivs-koy/DM15HPnKRrKuHNyKt6wUGg'}, {'nom': 'Le Repaire De Bacchus', 'url': 'https://www.ubereats.com/fr/store/le-repaire-de-bacchus-convention-paris/IXd9Vy2aUZGlq4JlWPjksw'}, {'nom': 'Yithé bubble tea shop 以茶.红峦', 'url': 'https://www.ubereats.com/fr/store/yithe-bubble-tea-shop-以茶红峦/XgcR96kUT6myjl17Wh8Ekg'}, {'nom': 'Ocho', 'url': 'https://www.ubereats.com/fr/store/ocho/28J4kLO9UGGHO-65spyVcQ'}, {'nom': 'Noom Tea St-Germain – Bubble Tea', 'url': 'https://www.ubereats.com/fr/store/noom-tea-st-germain-–-bubble-tea/6WyJo1eLQp-YdyWxe_D1qg'}, {'nom': 'Pizza Fidélité', 'url': 'https://www.ubereats.com/fr/store/pizza-fidelite/WPNs8rzeSr-Adl4z6-1-xw'}, {'nom': 'Sakura', 'url': 'https://www.ubereats.com/fr/store/sakura-saint-maurice/0RqFk9FxRQiTQvufisU2NA'}, {'nom': 'Louie Louie', 'url': 'https://www.ubereats.com/fr/store/louie-louie/WuB6CiuiVHmFdixyW_QyJQ'}, {'nom': 'Mangez et Cassez-Vous!', 'url': 'https://www.ubereats.com/fr/store/mangez-et-cassez-vous!-faidherbe/eu_hmBT4UM6ZB3xv26F-MA'}, {'nom': 'Aloy Thaï', 'url': 'https://www.ubereats.com/fr/store/aloy-thai-ivry/kMtF1ktzRDuo5DbqzArw0A'}, {'nom': 'FABBREZZA 🍕', 'url': 'https://www.ubereats.com/fr/store/fabbrezza/JDO673YyXsuEN4aS5GNZig'}, {'nom': 'Au Régal Paris', 'url': 'https://www.ubereats.com/fr/store/au-regal-paris/HxIywipQUAiGob76F1sf0g'}, {'nom': 'MyFrench Factory', 'url': 'https://www.ubereats.com/fr/store/myfrench-factory/b0zF0HYRTGyNCvL3RHT7OA'}, {'nom': 'Le 129 © Officiel', 'url': 'https://www.ubereats.com/fr/store/le-129-officiel-vitry/sCzbKUo-WZy7Q8YoxbgeIQ'}, {'nom': 'Eat Chicken', 'url': 'https://www.ubereats.com/fr/store/eat-chicken-montrouge/8FhrHAuDVxiITKnA5ANS3A'}, {'nom': 'WILAYA DZ CANTINE 🇩🇿', 'url': 'https://www.ubereats.com/fr/store/wilaya-dz-cantine/L7ODBoGaQHyhr02dUZz0zQ'}, {'nom': 'OBTB', 'url': 'https://www.ubereats.com/fr/store/obtb-paris-5/Swy-mXNBVnu_5CP4CWXqKQ'}, {'nom': '100% Crousti Original', 'url': 'https://www.ubereats.com/fr/store/100-crousti-original-paris-9/8M3SAkK_WhiB--RnOs5F2Q'}, {'nom': 'Nem Star', 'url': 'https://www.ubereats.com/fr/store/nem-star/AHjx5bPfSqGk_vO2RqElpQ'}, {'nom': 'Culture Crêpes', 'url': 'https://www.ubereats.com/fr/store/culture-crepes/z7IqA5mfRJW9_lmO0I7Acw'}, {'nom': 'Grill Istanbul', 'url': 'https://www.ubereats.com/fr/store/grill-istanbul/KeMUPkycUXqjANrZssmKzA'}, {'nom': 'ZAEKA Tandoori', 'url': 'https://www.ubereats.com/fr/store/zaeka-tandoori/8IgEWJM5T4e9cYngR1isSg'}, {'nom': "Aujourd'hui Demain", 'url': 'https://www.ubereats.com/fr/store/aujourdhui-demain/edoy5Z5QQwym7fVYF-BZKg'}, {'nom': 'Gfc paris 13', 'url': 'https://www.ubereats.com/fr/store/gfc-paris-13/uK8fPqDBQRCchiuH4k6FbQ'}, {'nom': 'Kallu & Co', 'url': 'https://www.ubereats.com/fr/store/kallu-%26-co/7-wAUVDcTb-Z5InRAdt78g'}, {'nom': 'Arya Bhavan', 'url': 'https://www.ubereats.com/fr/store/arya-bhavan/b_XT0sGWR8ePCOnXvIEhKQ'}, {'nom': 'Cupkie', 'url': 'https://www.ubereats.com/fr/store/cupkie-cookie-dough-a-la-francaise/slXhxwNWQgy8gadPXKpPqA'}, {'nom': 'Nomás', 'url': 'https://www.ubereats.com/fr/store/nomas-paris-2-mexican-street-food/cIZCxADnUNiZqPlbSahKIg'}, {'nom': 'Best Crêpes et Waffles', 'url': 'https://www.ubereats.com/fr/store/best-crepes-et-waffles/PMyyAvVxSmm5AnJt413fMw'}, {'nom': 'Rolling Naan', 'url': 'https://www.ubereats.com/fr/store/rolling-naan/w79bnTWSS3ShrHRJDF_FHw'}, {'nom': 'Chicken Spot', 'url': 'https://www.ubereats.com/fr/store/chicken-spot-paris-5/3OUoGAOEQOmFzb9JiOIXig'}, {'nom': 'Crousty Vice', 'url': 'https://www.ubereats.com/fr/store/crousty-vice/jTbN4LnWX5-jTYJCwPfRCA'}, {'nom': 'Mezo Grill', 'url': 'https://www.ubereats.com/fr/store/mezo-grill/skOEaOmIRRefjBvLRx0-cA'}, {'nom': 'Kebab & Tacos Factory 🥙', 'url': 'https://www.ubereats.com/fr/store/kebab-%26-tacos-factory/QXiodyrNXIGX9cDy4TE0hQ'}, {'nom': 'HOT & CROUSTY', 'url': 'https://www.ubereats.com/fr/store/hot-%26-crousty/yDZuVnD1VEmk8V6z5tv0Sg'}, {'nom': 'Crousty Box Paris 13', 'url': 'https://www.ubereats.com/fr/store/crousty-box-paris-13/3kiDlFtFXkmO_aYw6JwRCQ'}, {'nom': 'Phô Awl Montreuil', 'url': 'https://www.ubereats.com/fr/store/pho-awl-montreuil/vsLns1GyR6KbDw_tpUnLHA'}, {'nom': 'Suraj', 'url': 'https://www.ubereats.com/fr/store/suraj/XYWTlfmARZS9IH32sKQLxA'}, {'nom': 'Delicious social club', 'url': 'https://www.ubereats.com/fr/store/delicious-social-club/xluZYlJ2V2O8DgOxpokinQ'}, {'nom': 'crousty go', 'url': 'https://www.ubereats.com/fr/store/crousty-go/TonHwaAMR7KgrD193lzRbA'}, {'nom': 'Amorino', 'url': 'https://www.ubereats.com/fr/store/amorino-vosges/Jl94z5i7TQGp0ri7HqTeWA'}, {'nom': "C'est Mon Donuts- Italie 2", 'url': 'https://www.ubereats.com/fr/store/cest-mon-donuts-italie-2/zoGcbzUcRciURgF1VqYbXQ'}, {'nom': 'Didouw Cooks', 'url': 'https://www.ubereats.com/fr/store/didouw-cooks/JQWLHxx_QomYgeZ1tO3tdQ'}, {'nom': 'Boulangerie Copains Faubourg Saint-Antoine 🥐', 'url': 'https://www.ubereats.com/fr/store/boulangerie-copains-faubourg-saint-antoine/0JE86lFnWIGEhfg9grUjKw'}, {'nom': 'Amore e Gusto', 'url': 'https://www.ubereats.com/fr/store/amore-e-gusto-paris-14e/OnCRh2T-W02LlhIKlwFzBw'}, {'nom': 'CRISPY BANGER 🍚🐔💥 Crousty', 'url': 'https://www.ubereats.com/fr/store/crispy-banger-crousty-batignolles/ojv39mXdUraps6ETLlVnAg'}, {'nom': 'Master 🐓 Poulet', 'url': 'https://www.ubereats.com/fr/store/master-poulet-oberkampf/tzmB7U1jXfa5yi2Kv4aIdw'}, {'nom': 'ASADO Carne Argentina', 'url': 'https://www.ubereats.com/fr/store/asado-carne-argentina/xDQKxfv_UJyvuvn85zuYsw'}, {'nom': 'PITA Paris Opéra 🍋', 'url': 'https://www.ubereats.com/fr/store/pita-paris-opera/eNipQwNlUx28hUMAM0kIBg'}, {'nom': 'Indians Bowls', 'url': 'https://www.ubereats.com/fr/store/indians-bowls-paris/qrOgYMOgQK2uKGmId0Ux0A'}, {'nom': 'Dhasan', 'url': 'https://www.ubereats.com/fr/store/dhasan-bastille/hfY0jtPTW7K82IFs_vDXvQ'}, {'nom': 'GRILL MAÎTRE POULET', 'url': 'https://www.ubereats.com/fr/store/grill-maitre-poulet/ZseEdKXATja3yWkJbJOodw'}, {'nom': 'Best of India Tolbiac', 'url': 'https://www.ubereats.com/fr/store/best-of-india-tolbiac/_bI-bXsKUvCo-p9HLQE2WQ'}, {'nom': 'Léon', 'url': 'https://www.ubereats.com/fr/store/leon-chatelet/HEYUoAfMRZ-FZgAGH6Nb7g'}, {'nom': "Sonny's Pizza Strasbourg Saint Denis", 'url': 'https://www.ubereats.com/fr/store/sonnys-pizza-strasbourg-saint-denis/9k2Af8bkWc-rIOFmeZKAQQ'}, {'nom': 'Lily Pâtes 川里川外担担面', 'url': 'https://www.ubereats.com/fr/store/lily-pates-川里川外担担面/5V7f4tMNXoGA1VWj9giwog'}, {'nom': 'Nagano', 'url': 'https://www.ubereats.com/fr/store/nagano/lTH_x1H5SLC0hxamVLh0tA'}, {'nom': 'Garba Chaud', 'url': 'https://www.ubereats.com/fr/store/garba-chaud-la-courneuve/WN9Mlp71XFmpQhVELUllrA'}, {'nom': 'Funky Bao', 'url': 'https://www.ubereats.com/fr/store/funky-bao/bhXM8ZSbT7uISAYRb5bTpg'}, {'nom': 'Subway®', 'url': 'https://www.ubereats.com/fr/store/subway-chatelet/SZQ0mmEtWOKmJLcMv91jgw'}, {'nom': 'Frichti', 'url': 'https://www.ubereats.com/fr/store/frichti-paris-12/JxBcyFKQUH6xpKWkTb4fMw'}, {'nom': 'Yatai Ramen Montparnasse', 'url': 'https://www.ubereats.com/fr/store/yatai-ramen-montparnasse/FqI1itB6VdmfXymYkrQWIw'}, {'nom': 'Gaijin Ramen', 'url': 'https://www.ubereats.com/fr/store/gaijin-ramen-bastille/3DSgPD28VBmrdBpdRC12Ag'}, {'nom': 'Yaki', 'url': 'https://www.ubereats.com/fr/store/yaki/bO5JWvRHRJ6Pi7SQnDLPUg'}, {'nom': 'Becco 🍕 Pizza Romaine By Batistou', 'url': 'https://www.ubereats.com/fr/store/becco-pizza-romaine-by-batistou-pere-lachaise/PmXe8h-jXQmE1Yb69CmOVw'}, {'nom': 'Di PASTA', 'url': 'https://www.ubereats.com/fr/store/di-pasta/ecrk_ehXR5Wjdt5YlMhUyQ'}, {'nom': 'Baba Bey', 'url': 'https://www.ubereats.com/fr/store/baba-bey/cN2D6t-bRVKskBAyXEcfoA'}, {'nom': 'Bangkok Factory', 'url': 'https://www.ubereats.com/fr/store/bangkok-factory-gambetta/EP8TcsDYRga90mgO2o87Jw'}, {'nom': 'Maître Naan', 'url': 'https://www.ubereats.com/fr/store/maitre-naan-paris-14-eme/BbBxXjQTX6ikwfj-RQAFyg'}, {'nom': 'Okito', 'url': 'https://www.ubereats.com/fr/store/okito-convention/chBcYjJdRge5R_719PeqFg'}, {'nom': 'Grill Gap', 'url': 'https://www.ubereats.com/fr/store/grill-gap/OFKSF7pOQ72hC8vyFhH6Ow'}, {'nom': "Sonny's Pizza", 'url': 'https://www.ubereats.com/fr/store/sonnys-pizza/I5u-pYWlSrK9nt-rRLhfjw'}, {'nom': 'Edo Sushi', 'url': 'https://www.ubereats.com/fr/store/edo-sushi/7AmS9p4cQfuR426aixe2ZA'}, {'nom': 'Rivières D’Himalaya', 'url': 'https://www.ubereats.com/fr/store/rivieres-dhimalaya/XrsESG7sR5mKY6VNzj6kiA'}, {'nom': 'Keko Momo 馍面坊', 'url': 'https://www.ubereats.com/fr/store/keko-momo-馍面坊/U5ZRa1CwURaMiHbeSMmqdw'}, {'nom': 'Pasta District 18', 'url': 'https://www.ubereats.com/fr/store/pasta-district-18/6rJXMpO9Q0Sdp7DIBrOpRg'}, {'nom': 'Bowlywood', 'url': 'https://www.ubereats.com/fr/store/bowlywood-bastille/F8DRNPUoWe6ft0Z9Mw-nTw'}, {'nom': 'BARAQUE 🍟 Frites bowl', 'url': 'https://www.ubereats.com/fr/store/baraque-frites-bowl-champigny-sur-marne/YBFSnWWDXxeH5H4bcd9_Tw'}, {'nom': 'Wukong Temple', 'url': 'https://www.ubereats.com/fr/store/wukong-temple/Fu623J-hVQuRwB6ZEE8TnQ'}, {'nom': 'Poke House', 'url': 'https://www.ubereats.com/fr/store/poke-house-forum-des-halles/SKXni4JKVUGsLQNMH4EGDQ'}, {'nom': 'Dalmata', 'url': 'https://www.ubereats.com/fr/store/dalmata-paris-10/iBNBUu5GSt-IDU3Lv7Y9UQ'}, {'nom': 'New Art du Ravioli 🥟', 'url': 'https://www.ubereats.com/fr/store/new-art-du-ravioli/xVQUWcqSRG6nsXnqUmZr1w'}, {'nom': "Monop'", 'url': 'https://www.ubereats.com/fr/store/monop-madeleine/jCFiRd_8QBaeOu1SzFfbjA'}, {'nom': 'Nagoya Vouillé', 'url': 'https://www.ubereats.com/fr/store/nagoya-vouille/i5Qs8yuvU2iAh_LVfSpOew'}, {'nom': 'Bibigo 🇰🇷 Riz Poulet Crousty Coréen', 'url': 'https://www.ubereats.com/fr/store/bibigo-riz-poulet-crousty-coreen-bastille/sa4RGws2Rjyjj5_6iQBSBg'}, {'nom': 'Yooki Ramen', 'url': 'https://www.ubereats.com/fr/store/yooki-ramen/E4bNh5XNWAe7sVL6_vcDzQ'}, {'nom': 'Petit Bao Faubourg Saint Denis', 'url': 'https://www.ubereats.com/fr/store/petit-bao-faubourg-saint-denis/tpJbG1a5WLCWLhPhztwDWA'}, {'nom': 'Madras Bhavan', 'url': 'https://www.ubereats.com/fr/store/madras-bhavan/dS-EnlbfSsWwZtvHoy_ZzQ'}, {'nom': 'Sushi Muraguchi', 'url': 'https://www.ubereats.com/fr/store/sushi-muraguchi/RpUmxx7xRNmgnGib5FwNzQ'}, {'nom': 'Seoul Gang 🍗', 'url': 'https://www.ubereats.com/fr/store/seoul-gang/PrVlbE57WuKEt7B62g97wg'}, {'nom': 'Dumbo Petites Ecuries', 'url': 'https://www.ubereats.com/fr/store/dumbo-petites-ecuries/IwaiYfC9XKaZ1DoEHf4yoA'}, {'nom': 'Mealy by Michou 🍔', 'url': 'https://www.ubereats.com/fr/store/mealy-by-michou-paris/OkGp5Qf8Xligj4Z4P09FDQ'}, {'nom': 'El Guacamole', 'url': 'https://www.ubereats.com/fr/store/el-guacamole/TRqC_2vYTHWm4MIC_m2AKA'}, {'nom': 'Orgasmeat', 'url': 'https://www.ubereats.com/fr/store/orgasmeat-hopital-saint-louis/a4sybp4NVHeJ6F_f5dQ6DQ'}, {'nom': 'THAIGER', 'url': 'https://www.ubereats.com/fr/store/thaiger/iimVofonVJyLmvqXGcpCzw'}, {'nom': 'Ngoc Xuyen Saigon', 'url': 'https://www.ubereats.com/fr/store/ngoc-xuyen-saigon/BPid3CPASb-sKPhlZ4T51Q'}, {'nom': 'Indian Maan Mahal', 'url': 'https://www.ubereats.com/fr/store/indian-maan-mahal/GIIQiYsyQBiMQfU3Qwj7ZA'}, {'nom': 'LITTLE ITALIA', 'url': 'https://www.ubereats.com/fr/store/little-italia/R3Z7Rte7XNSgzihoYxjNJg'}, {'nom': 'Clasico Argentino Empanadas 🇦🇷  Montrouge', 'url': 'https://www.ubereats.com/fr/store/clasico-argentino-empanadas-montrouge/HbyYCFOXVfaTcuRdJB_ASA'}, {'nom': 'Jean sushi', 'url': 'https://www.ubereats.com/fr/store/jean-sushi/-k4IL-BjTc2iu0OiqIH3CA'}, {'nom': 'Naturalia', 'url': 'https://www.ubereats.com/fr/store/naturalia-vincennes-chateau/c7rhEycJU0edFIcil17lqg'}, {'nom': 'Uncle Bumpy', 'url': 'https://www.ubereats.com/fr/store/uncle-bumpy-arcueil/YkbHpYAaTnyTj9cA4EhMCA'}, {'nom': 'Crousty Crispy', 'url': 'https://www.ubereats.com/fr/store/crousty-crispy-les-halles/hjB5ltAbVK2LppRL2-U0ig'}, {'nom': 'Vapiano', 'url': 'https://www.ubereats.com/fr/store/vapiano-champs-elysees/m7niXGZvROO4RinvPKe6WQ'}, {'nom': 'Le Caribéen Paris 10', 'url': 'https://www.ubereats.com/fr/store/le-caribeen-paris-10/hNspzj17RfusFE1R3xfiPw'}, {'nom': 'Nysa cavistes Aligre', 'url': 'https://www.ubereats.com/fr/store/nysa-cavistes-aligre-vins-bieres-%26-spiritueux/beyArVHxRtGbt0NdsVvDVg'}, {'nom': 'Wad Thai', 'url': 'https://www.ubereats.com/fr/store/wad-thai/k2VFbZKrQwqpjyxnls0E0Q'}, {'nom': 'Au Piment Du Centre 蜀园', 'url': 'https://www.ubereats.com/fr/store/au-piment-du-centre-蜀园/KHbBwF3iTlKplqINyeitlg'}, {'nom': 'New Aqib', 'url': 'https://www.ubereats.com/fr/store/new-aqib-restaurant-indien/LhQlHw-8T5KhaLnxkIFCyw'}, {'nom': 'Pharmacie varenne by Nexum', 'url': 'https://www.ubereats.com/fr/store/pharmacie-varenne-by-nexum/TLEcvC8sVaqKKAyvCkg0Zw'}, {'nom': 'Mme Shawn Caffarelli', 'url': 'https://www.ubereats.com/fr/store/mme-shawn-caffarelli/2rNKXuMtWcirw5IcUCC06w'}, {'nom': 'Crousty Game', 'url': 'https://www.ubereats.com/fr/store/crousty-game-paris-18/hEw92u2aWT2KQ04NJteDWw'}, {'nom': 'Tricotin', 'url': 'https://www.ubereats.com/fr/store/tricotin/nrd_kY_UXPmCTagDvXNSjg'}, {'nom': 'Peppe Pizzeria', 'url': 'https://www.ubereats.com/fr/store/peppe-pizzeria-paris-20/DMetNmmbQUuVSgH7uUbXzA'}, {'nom': '🔥 HANWOO Haussmann 🔥', 'url': 'https://www.ubereats.com/fr/store/hanwoo-haussmann/8XV990WgTFaeZ6h0pZTpgA'}, {'nom': 'Sakura', 'url': 'https://www.ubereats.com/fr/store/sakura/ujtSpUbVS1m_8GWDkfbAcA'}, {'nom': 'Restaurant ETHIOPIA', 'url': 'https://www.ubereats.com/fr/store/restaurant-ethiopia/edw5nzHTQs2GacpAXuVg6g'}, {'nom': 'CRAZY NOODLES 西北疯', 'url': 'https://www.ubereats.com/fr/store/crazy-noodles-西北疯/flPMR-o2UP2mxVMr-PVMHg'}, {'nom': "Saveurs d'Himalaya", 'url': 'https://www.ubereats.com/fr/store/saveurs-dhimalaya/CmLoZJEDTjmKvMblJYcHMA'}, {'nom': 'XI’an Dining 陕食堂', 'url': 'https://www.ubereats.com/fr/store/xian-dining-陕食堂/4ocwZbeoUVSZN5Jgh8QpVg'}, {'nom': 'Gangnam', 'url': 'https://www.ubereats.com/fr/store/gangnam/NEU36SE-VDeADPKzZuwhOg'}, {'nom': 'Wisteria Fleuriste 💐', 'url': 'https://www.ubereats.com/fr/store/wisteria-fleuriste/EPMbeiS7WAu3yD2BlL05uQ'}, {'nom': 'Hokkaido', 'url': 'https://www.ubereats.com/fr/store/hokkaido/hqG-9feMRcqQhUZfoE07_A'}, {'nom': 'Yeko', 'url': 'https://www.ubereats.com/fr/store/yeko/H4_Hdg8aUkO1IhQuVG_wTw'}, {'nom': 'DNoodles-oberkampf娣小面', 'url': 'https://www.ubereats.com/fr/store/dnoodles-oberkampf娣小面/9wzsH2XWWBCkzQUpIBN9WA'}, {'nom': 'Madama', 'url': 'https://www.ubereats.com/fr/store/madama/YiyHP00hQ-qPveu05RFFuA'}, {'nom': 'O.B.U. Organisation des Burgers Unis', 'url': 'https://www.ubereats.com/fr/store/obu-organisation-des-burgers-unis-vavin/ZJwAb8qETsmJ8XjIr7RMww'}, {'nom': 'Hippopotamus', 'url': 'https://www.ubereats.com/fr/store/hippopotamus-paris-montparnasse-14e/uRQxKzmHTJWU-mzIJhdOYA'}, {'nom': 'SUMO', 'url': 'https://www.ubereats.com/fr/store/sumo-sas-yu/HEA0DKInXSyOlJSpa7Ft2w'}, {'nom': 'La Brigade', 'url': 'https://www.ubereats.com/fr/store/la-brigade-opera/TuEa-zrfSu62r3m7lx7O4w'}, {'nom': "Afrik'N'Fusion", 'url': 'https://www.ubereats.com/fr/store/afriknfusion-20e/nqTJdIBiTGS_MgY-KDkxyw'}, {'nom': 'Planet Sushi', 'url': 'https://www.ubereats.com/fr/store/planet-sushi-issy-les-moulineaux/rV9ftNHxUlamLsoOu3qjIw'}, {'nom': 'Karamen', 'url': 'https://www.ubereats.com/fr/store/karamen-gentilly/kNUiAkrKV9OYIJVQp8Z4bQ'}, {'nom': 'KARE Curry & Donburi', 'url': 'https://www.ubereats.com/fr/store/kare-curry-%26-donburi/sQuzjZ9FUvysdp2D4TwHAQ'}, {'nom': 'Thai Kamkoun', 'url': 'https://www.ubereats.com/fr/store/thai-kamkoun/GVVAXTtMWtmjEkHNlAuWbA'}, {'nom': 'ACÀ REPUBLIQUE', 'url': 'https://www.ubereats.com/fr/store/aca-republique/zN40GQRoU4-eZm9tjbzrSw'}, {'nom': 'Le Petit Japonais', 'url': 'https://www.ubereats.com/fr/store/le-petit-japonais/L70sHh6sRSqrCo2dReH0sA'}, {'nom': 'Istanbul Grill', 'url': 'https://www.ubereats.com/fr/store/istanbul-grill/CfYinG3xTladtL9N9t7cBg'}, {'nom': "Shawar'Mama", 'url': 'https://www.ubereats.com/fr/store/shawarmama-paris-16/8icxcVngWYSBV8741-9EGA'}, {'nom': 'Doki Doki', 'url': 'https://www.ubereats.com/fr/store/doki-doki-louvre/ZhsyMHROQj-0iw0xHBIYkw'}, {'nom': 'Burger King', 'url': 'https://www.ubereats.com/fr/store/burger-king-paris-opera-italiens/_2kSRPPERUSGEK4ppJHnmw'}, {'nom': 'Délices de la Croix Nivert', 'url': 'https://www.ubereats.com/fr/store/delices-de-la-croix-nivert/FST-364ASkqyFvDTwqjXvQ'}, {'nom': 'Homer Lobster 🦞', 'url': 'https://www.ubereats.com/fr/store/homer-lobster-marais/5CVnjAcYXfqt7ThusQoePA'}, {'nom': 'The Hood Paris', 'url': 'https://www.ubereats.com/fr/store/the-hood-paris/YwMpYqQzTRGTsIPueF4bgw'}, {'nom': 'Lucky Box', 'url': 'https://www.ubereats.com/fr/store/lucky-box/hM1ZT7JWQFSypGrkMtishQ'}, {'nom': 'Pure Cuisine', 'url': 'https://www.ubereats.com/fr/store/pure-cuisine-montreuil/ToveIuwzQyCvpxi4MCf4tA'}, {'nom': 'La Bonne Graye', 'url': 'https://www.ubereats.com/fr/store/la-bonne-graye-malakoff/O5Mj_EN_VhuiYgAHGKBIiA'}, {'nom': 'Karachi Biryani', 'url': 'https://www.ubereats.com/fr/store/karachi-biryani/m2pb6bZUVf-UvF3vvsptDQ'}, {'nom': 'La Pause Libanaise', 'url': 'https://www.ubereats.com/fr/store/la-pause-libanaise-sentier/5kIgSEx3Uwy3NCBd_S_0cw'}, {'nom': 'Asahi', 'url': 'https://www.ubereats.com/fr/store/asahi/IwATgbwJS5K-WhkcTpVmcQ'}, {'nom': 'Sushi Passy', 'url': 'https://www.ubereats.com/fr/store/sushi-passy/wlj4_c4uVLSdM4pAP5qnbQ'}, {'nom': 'Como KITCHEN', 'url': 'https://www.ubereats.com/fr/store/como-kitchen-republique/T5niy9_JSWuOJV65V7i-Nw'}, {'nom': '三羊开泰 Paris yum', 'url': 'https://www.ubereats.com/fr/store/三羊开泰-paris-yum/VBxYN2q7UBCvGrkxY4u0OA'}, {'nom': 'Vafamoc Pizza frite Napolitaine', 'url': 'https://www.ubereats.com/fr/store/vafamoc-pizza-frite-napolitaine/5_reCL-IVymsmv8jKj5q9Q'}, {'nom': 'Gandhi Restaurant', 'url': 'https://www.ubereats.com/fr/store/gandhi-restaurant/AgbdjmK2Tl-7JVXrlCGs4g'}, {'nom': 'Asian Kitchen', 'url': 'https://www.ubereats.com/fr/store/asian-kitchen/IaYmUD5JRZ67Yu_Ij-SvDQ'}, {'nom': 'BARAK', 'url': 'https://www.ubereats.com/fr/store/barak-poissonniere/Ot_N-mDNQ3mUZPmjks9XQA'}, {'nom': 'Berliner Das Original', 'url': 'https://www.ubereats.com/fr/store/berliner-das-original-kebab-berlinois-linne/Cq8AzTQDUY6eU4QDWvBN4A'}, {'nom': 'NACHOS Mexican Good Food', 'url': 'https://www.ubereats.com/fr/store/nachos-mexican-good-food-chatelet-les-halles/tCfBqiTFVn-BaF-aWmM5YA'}, {'nom': "Bil's Thai & Burger", 'url': 'https://www.ubereats.com/fr/store/bils-thai-%26-burger/l1GLspPwUci5xP328DqFGA'}, {'nom': "Chicken's King", 'url': 'https://www.ubereats.com/fr/store/chickens-king-paris-10eme/zNebJLnqRc6Tdh6HYcJgfw'}, {'nom': 'KFC', 'url': 'https://www.ubereats.com/fr/store/kfc-forum-des-halles/G_jPFz41U2WtQl6877_Yaw'}, {'nom': 'Le Soundiata', 'url': 'https://www.ubereats.com/fr/store/le-soundiata/VaQgGGwAROWGbo8V4iYo9Q'}, {'nom': 'Saphir Fleurs', 'url': 'https://www.ubereats.com/fr/store/saphir-fleurs-paris/2TZcxkyrV-mzSn_Q2wIcvQ'}, {'nom': 'Le Bortsch', 'url': 'https://www.ubereats.com/fr/store/le-bortsch/BFMHZTC-TKGXpRzibHwqvA'}, {'nom': "Bio C' Bon", 'url': 'https://www.ubereats.com/fr/store/bio-c-bon-paris-vavin/YydehvKtWa2o6_-1DcQtOA'}, {'nom': 'Cheese NAN', 'url': 'https://www.ubereats.com/fr/store/cheese-nan/kMSOoGZCSjebE3nSvVyvzQ'}, {'nom': 'Five Guys', 'url': 'https://www.ubereats.com/fr/store/five-guys-burgers-fries-%26-milkshakes-opera/OR7wL5fFXPGd8EOS_gST2A'}, {'nom': 'Papaya Chicken 🍗', 'url': 'https://www.ubereats.com/fr/store/papaya-chicken/PTerWEjFXlKRnvZs1g66RQ'}, {'nom': 'Chapeau Restaurant Iranien 🇮🇷', 'url': 'https://www.ubereats.com/fr/store/chapeau-restaurant-iranien/-8uZFzbVXSyfQBq2kW1P1g'}, {'nom': '🥗 POKE UP Paris 11 Poke Bowl & Bubble Tea', 'url': 'https://www.ubereats.com/fr/store/poke-up-paris-11-poke-bowl-%26-bubble-tea/6UAktrG9WSyPVq3GgCDu1g'}, {'nom': "Au P'tit Crêpe", 'url': 'https://www.ubereats.com/fr/store/au-ptit-crepe/DQkhiieWSVGXWl88maC9bA'}, {'nom': 'Chez Céline', 'url': 'https://www.ubereats.com/fr/store/chez-celine/6nMH5SmTUhKoP8xKBagaGA'}, {'nom': 'Mangez et Cassez-Vous ! Taitbout', 'url': 'https://www.ubereats.com/fr/store/mangez-et-cassez-vous-!-taitbout/w0yrQxNRUk6ohqY1R1aGuA'}, {'nom': 'Les Fondus de la Raclette', 'url': 'https://www.ubereats.com/fr/store/les-fondus-de-la-raclette-parmentier/i9DG2jRgWSWWU-vldqkvBQ'}, {'nom': 'Pasta Machine Tradition 🍝', 'url': 'https://www.ubereats.com/fr/store/pasta-machine-tradition-rivoli/qAG7eAIKURmNrTmnxRFj8Q'}, {'nom': 'La Grosse Commande 13ème', 'url': 'https://www.ubereats.com/fr/store/la-grosse-commande-13eme-le-food-court-des-commandes-a-plusieurs/AsGHr787X7KU2NFFv8B90w'}, {'nom': 'Mavrommatis', 'url': 'https://www.ubereats.com/fr/store/mavrommatis-censier/9ipmMxCGS_CmzR9LV4OAtw'}, {'nom': 'Ala-Din Resto', 'url': 'https://www.ubereats.com/fr/store/ala-din-resto-kebab/lVXamnZjV1eppADjF4DV4Q'}, {'nom': 'Le Petit Phénicien', 'url': 'https://www.ubereats.com/fr/store/le-petit-phenicien/dz-ccHyyTm-YxeBNcfHuSQ'}, {'nom': 'Salade Power', 'url': 'https://www.ubereats.com/fr/store/salade-power-rivoli/vmrgIBSnXhWcGqVDzaYg2Q'}, {'nom': 'PEPEGUSTO 🍕❤️', 'url': 'https://www.ubereats.com/fr/store/pepegusto/jWM0p1W8Vd-kd6g_ljzvaw'}, {'nom': 'Machi Machi Marais 麥吉', 'url': 'https://www.ubereats.com/fr/store/machi-machi-marais-麥吉/GmjvrPogXyK7UwvQo1vrRg'}, {'nom': 'Out Fry Korean Fried Chicken', 'url': 'https://www.ubereats.com/fr/store/out-fry-korean-fried-chicken-tolbiac/4wwEyDL5Vey4PGZpPrvGKw'}, {'nom': 'MELT DELI', 'url': 'https://www.ubereats.com/fr/store/melt-deli-les-halles/OtaDJ1_oUhOxObUQfxvM6w'}, {'nom': 'Dangerous Burger', 'url': 'https://www.ubereats.com/fr/store/dangerous-burger-bastille/ZIGR69isWTKcg3phCXknPw'}, {'nom': 'New Bistrot Tokyo', 'url': 'https://www.ubereats.com/fr/store/new-bistrot-tokyo/0QFUajvmRY2Sdk1HV2JRDw'}, {'nom': 'Dolce Pizza', 'url': 'https://www.ubereats.com/fr/store/dolce-pizza/q6-gDrj0Q2iFFofULOclUg'}, {'nom': "Le Saigon D'antan", 'url': 'https://www.ubereats.com/fr/store/le-saigon-dantan/8JIlWRIITlmHjPNrCmtoJw'}, {'nom': "Schwartz's Hot Dog Hôtel de Ville", 'url': 'https://www.ubereats.com/fr/store/schwartzs-hot-dog-hotel-de-ville/j_tgLrYyQ72pq_OQ7gZ-4Q'}, {'nom': 'Smash Burger & Fries Eiffel', 'url': 'https://www.ubereats.com/fr/store/smash-burger-%26-fries-eiffel/Eb7-YPKmU46LBfTpn3uzNQ'}, {'nom': 'Amore Mio', 'url': 'https://www.ubereats.com/fr/store/amore-mio/L_uBvb_4T6aqGJtcHp_ybg'}, {'nom': 'MAMÉ Kitchen', 'url': 'https://www.ubereats.com/fr/store/mame-kitchen-poissonniere/wQDrY6OeXeyEJ8Y8B3Z61A'}, {'nom': 'Krousty Furai', 'url': 'https://www.ubereats.com/fr/store/krousty-furai/zDBMtOJjQQGTK98LFVmtxA'}, {'nom': 'Krousty Panda🐼', 'url': 'https://www.ubereats.com/fr/store/krousty-panda-paris-pereire/O7NWBvU1WSeun-HdbdE40A'}, {'nom': "Jap'n", 'url': 'https://www.ubereats.com/fr/store/japn-porte-saint-martin/314sa9h5VYaf3RJDGpSioQ'}, {'nom': 'Smash World', 'url': 'https://www.ubereats.com/fr/store/smash-world-bagneux/6OyfOfWVSBCclj67r0RvCA'}, {'nom': 'Le Wok Saint-Germain', 'url': 'https://www.ubereats.com/fr/store/le-wok-saint-germain/uj1M4BfSRd-EGYiKEq5RdQ'}, {'nom': 'Restaurant Ella', 'url': 'https://www.ubereats.com/fr/store/restaurant-ella/tv1af2qcQ76vWYpw5z_hSA'}, {'nom': "Le Parfum De L'Inde 🇮🇳", 'url': 'https://www.ubereats.com/fr/store/le-parfum-de-linde/HdRudXXQQRK4ym5mzRsPqg'}, {'nom': 'SCALINI PARIS', 'url': 'https://www.ubereats.com/fr/store/scalini-paris/UCyE8xwkXdyK4vnV6P8Gtg'}, {'nom': 'KONAPOKE', 'url': 'https://www.ubereats.com/fr/store/konapoke-poke-%26-bowls-suffren/zkbDBi8OUMmS-sVHKZEtrQ'}, {'nom': 'Genki', 'url': 'https://www.ubereats.com/fr/store/genki-fontenay/0HSmbfZZVbSey4XLAhl04A'}, {'nom': 'Bollynan', 'url': 'https://www.ubereats.com/fr/store/bollynan-grands-boulevards/lzbtr0SURqia70U4HO26QA'}, {'nom': 'Just Smash', 'url': 'https://www.ubereats.com/fr/store/just-smash/BAFyM8j2WeSUGVZJLZ7aaA'}, {'nom': 'Liza Cuisine Libanaise', 'url': 'https://www.ubereats.com/fr/store/liza-cuisine-libanaise/jVu9plsXRdq7x8KP87TwuA'}, {'nom': 'FJM BERACA', 'url': 'https://www.ubereats.com/fr/store/fjm-beraca/Ugdb-P30Ryqzg8C2f2RqNQ'}, {'nom': 'Matsuyama Cadet', 'url': 'https://www.ubereats.com/fr/store/matsuyama-cadet/Ua9ODcelXjKDPfbgWj3Z0w'}, {'nom': 'Kiss Kiss Bang Bang', 'url': 'https://www.ubereats.com/fr/store/kiss-kiss-bang-bang/Ad99GuPxRIiyYPcYRveQ4Q'}, {'nom': 'PNY OBERKAMPF', 'url': 'https://www.ubereats.com/fr/store/pny-oberkampf/oL7YekniV1-n-4kztcpthw'}, {'nom': 'Osè African Cuisine', 'url': 'https://www.ubereats.com/fr/store/ose-african-cuisine-voltaire/n-M2CiKMWtmaLr0vA6b39w'}, {'nom': 'FnF Restaurant', 'url': 'https://www.ubereats.com/fr/store/fnf-restaurant/6_CZm0TiRK2P5zcumIGIGg'}, {'nom': 'Toyotomy', 'url': 'https://www.ubereats.com/fr/store/toyotomy/LMjJAZVjSGex_-sACpr2_g'}, {'nom': 'Bubble Bae 💘', 'url': 'https://www.ubereats.com/fr/store/bubble-bae-paris/5mvYDJahTeaQpg1rE3LyxA'}, {'nom': 'Minori', 'url': 'https://www.ubereats.com/fr/store/minori/U5ZixcVuQ0Odh4AupY7Dyg'}, {'nom': 'Idam', 'url': 'https://www.ubereats.com/fr/store/idam-paris-15/AfVDBYvaTcOe7gbOI1V8gg'}, {'nom': "Bill's Burger", 'url': 'https://www.ubereats.com/fr/store/bills-burger-chatelet/1OK3YcfMTvK3EBWDEIIM2A'}, {'nom': 'CUT Street Food 🔥', 'url': 'https://www.ubereats.com/fr/store/cut-street-food/sDAwqk7zStyM_E0D26U5gw'}, {'nom': 'Crousty Banger', 'url': 'https://www.ubereats.com/fr/store/crousty-banger-paris-10/QDr109zKX_uH0OEoyTnn_w'}, {'nom': 'Angelino Trattoria', 'url': 'https://www.ubereats.com/fr/store/angelino-trattoria/q2hK0pPLThuyUF8IvmjChA'}, {'nom': 'Al PUNJAB', 'url': 'https://www.ubereats.com/fr/store/al-punjab/sWp28kRGSV-qIULG2zM4PQ'}, {'nom': 'Khao Kitchen by Laosan', 'url': 'https://www.ubereats.com/fr/store/khao-kitchen-by-laosan-neuilly/zo9ASQtGTbCXDdcmirjULg'}, {'nom': 'Chicken Street', 'url': 'https://www.ubereats.com/fr/store/chicken-street-chatelet/jSWdsyV5VcyeDMxAX7Z7kg'}, {'nom': "Chicken's Time", 'url': 'https://www.ubereats.com/fr/store/chickens-time-vitry/3DJH5UP2W8mXYFMSsH2i1A'}, {'nom': 'Street Naan 🌯', 'url': 'https://www.ubereats.com/fr/store/street-naan/bR7AwdEKWmOjiG672gSeNg'}, {'nom': 'Yallah by Vendredi Soir', 'url': 'https://www.ubereats.com/fr/store/yallah-by-vendredi-soir/LpEdwIHYVAGX_SWPCL_lKA'}, {'nom': 'GOMU', 'url': 'https://www.ubereats.com/fr/store/gomu-ivry-sur-seine/V86H_xbAVamSWqHpx5LCWQ'}, {'nom': 'Krousty Factory', 'url': 'https://www.ubereats.com/fr/store/krousty-factory-ivry/Dzf0vQZtWdyv-sXjE1VicQ'}, {'nom': 'Yakitori', 'url': 'https://www.ubereats.com/fr/store/yakitori/HwEp0BwqT8Wr57YiJ1_0LA'}]


print(liste_des_restaurants_dans_la_categorie_actuelle)

[{'nom': "McDonald's®", 'url': 'https://www.ubereats.com/fr/store/mcdonalds-paris-louvre-rivoli/nWnUwvnuRBWFIRsBjSXlEQ'}, {'nom': 'Pizza Primavera 🍕', 'url': 'https://www.ubereats.com/fr/store/pizza-primavera-paris/HGHkqohYQ16XA7B2UhxuGA'}, {'nom': 'K-ViBE', 'url': 'https://www.ubereats.com/fr/store/k-vibe-fried-chicken-saint-honore/hi7Ox_x0WSGxgcq-vr1CKA'}, {'nom': 'DongNe Chicken', 'url': 'https://www.ubereats.com/fr/store/dongne-chicken/KQtl8gnrXYi1eG_cSl7GEQ'}, {'nom': 'Street Bangkok Odéon', 'url': 'https://www.ubereats.com/fr/store/street-bangkok-odeon/g5aEKN46SX26a3zEnz_Yvg'}, {'nom': 'Momidji', 'url': 'https://www.ubereats.com/fr/store/momidji/DDIiwB7HRgiS_X96vUN7Rw'}, {'nom': 'Krousty Sabaïdi', 'url': 'https://www.ubereats.com/fr/store/krousty-sabaidi-chatelet/2_5Q6WfmUKKb523t2wXP4A'}, {'nom': "O'Tacos", 'url': 'https://www.ubereats.com/fr/store/otacos-chatelet/iI439XtiRfSffGqy-aUJRw'}, {'nom': 'Chipotle Mexican Grill', 'url': 'https://www.ubereats.com/fr/store/chipotle-mexica

In [ ]:
#CODE POUR AJOUTER DIRECTEMENT DANS UN CSV LES INFO DU RESTAURANT

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import asyncio
import re
import csv
import os

# Exemple de ta liste
nom_colonne_par_defaut = [''] * 14
# A CHANGER PAR LA CATEGORIE ACTUELLE qui est scrap
nom_colonne_par_defaut[10] = "Asiatique"

# Variable globale
liste_final_restaurants_global = []

#  Création du fichier CSV avec le nom des colonnes
csv_filename = "asiatique.csv"
if not os.path.exists(csv_filename):
    with open(csv_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Nom restaurant","Adresse","Code postal","Nom plat","Prix","Description",
            "Section","", "Note","Avis","Catégorie fixée","Catégories","Prix catégorie",""
        ])

# Fonction qui prend en argument un lien vers un restaurant et le scrap
async def scrape_restaurant(url: str, p):
    global liste_final_restaurants_global

    plat = nom_colonne_par_defaut.copy()
    lst_plat = []

    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    try:
        await page.goto(url, timeout=60000)
        await page.wait_for_timeout(5000)

        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")

        # Les informations importantes sont dans la balise span
        info_spans = [span.get_text(strip=True) for span in soup.select("p span") if span.get_text(strip=True)]

        note, avis, categories, prix_categorie, adresse, code_postal = "", "", [], "", "", ""

        # Extraction infos générales sur le restaurant
        if info_spans and info_spans[0].replace('.', '', 1).isdigit():
            note = info_spans[0]
        if len(info_spans) > 3 and '(' in info_spans[3]:
            avis = info_spans[3]

        # Trouver les catégories
        try:
            start = info_spans.index('•') + 1
        except ValueError:
            start = None

        # Catégorie Prix
        if start:
            for i in range(start, len(info_spans)):
                val = info_spans[i]
                if val in ['€', '€€', '€€€', '$', '$$', '$$$', 'Informations']:
                    prix_categorie = val if '€' in val or '$' in val else None
                    break
                elif val != '•' and not val.startswith('x'):
                    categories.append(val)
            for val in reversed(info_spans):

          # Chercher un code postal à 5 chiffres
              match = re.search(r"\b\d{5}\b", val)

              if match:
                code_postal = match.group(0)
                adresse = val.strip()
                break


        if not adresse:
            print(f" Aucun adresse trouvée pour {url}, restaurant ignoré (hors Paris).")
            await browser.close()
            return None

        # Extraction du menu
        sections = soup.find_all("div", attrs={"data-testid": "store-catalog-section-vertical-grid"})
        for section in sections:

            section_name_tag = section.find("h3")
            section_name = section_name_tag.text.strip() if section_name_tag else "Section inconnue"

            items = section.find_all("li")
            for item in items:
                spans = item.find_all("span")
                if not spans:
                    continue

                name = spans[0].get_text(strip=True)
                price = spans[1].get_text(strip=True) if len(spans) > 1 else ""

                description = ""
                i = 2
                while i < len(spans):
                    txt = spans[i].get_text(strip=True)
                    if any(x in txt for x in ["€", "•", "%"]):
                        i += 1
                    else:
                        description = txt
                        break
                #Si on a récupérer le prix alors on ajoute le plat
                if name and "€" in price:

                    plat[0] = soup.find('h1').text.strip()
                    plat[1] = adresse
                    plat[2] = code_postal
                    plat[3] = name
                    plat[4] = price
                    plat[5] = description
                    plat[6] = section_name
                    plat[8] = note
                    plat[9] = avis
                    plat[11] = ", ".join(categories)
                    plat[12] = prix_categorie

                    # Stocke tous les plats
                    lst_plat.append(plat.copy())
                    liste_final_restaurants_global.append(plat.copy())

                    # Ajoute le plat dans le CSV
                    with open(csv_filename, "a", newline="", encoding="utf-8") as f:
                        writer = csv.writer(f)
                        writer.writerow(plat)

        await browser.close()
        return lst_plat

    except Exception as e:
        print(f" Erreur sur {url}: {e}")
        await browser.close()
        return None


async def main():
    async with async_playwright() as p:
      # Boucle sur les liens des restaurants dans la variable liste_des_restaurants_dans_la_categorie_actuelle
        for num_resto in range(0, 10):
            print("--- Restaurant " + str(num_resto) + " ---")
            print(f"\n Scraping {liste_des_restaurants_dans_la_categorie_actuelle[num_resto]['nom']} ...")
            data = await scrape_restaurant(liste_des_restaurants_dans_la_categorie_actuelle[num_resto]["url"], p)
            if data:
                print(f" {liste_des_restaurants_dans_la_categorie_actuelle[num_resto]['nom']} ({len(data)} plats trouvés)")
            else:
                print(f" {liste_des_restaurants_dans_la_categorie_actuelle[num_resto]['nom']} ignoré ou erreur.")

    print(f"\n--- FINI ---\nTotal plats collectés : {len(liste_final_restaurants_global)}")
    for elem in liste_final_restaurants_global[:5]:
        print(elem)


await main()


--- Restaurant 0 ---

 Scraping McDonald's® ...
⏭️ McDonald's® ignoré ou erreur.
--- Restaurant 1 ---

 Scraping Pizza Primavera 🍕 ...
✅ Pizza Primavera 🍕 (47 plats trouvés)
--- Restaurant 2 ---

 Scraping K-ViBE ...
✅ K-ViBE (69 plats trouvés)
--- Restaurant 3 ---

 Scraping DongNe Chicken ...
✅ DongNe Chicken (36 plats trouvés)
--- Restaurant 4 ---

 Scraping Street Bangkok Odéon ...
✅ Street Bangkok Odéon (29 plats trouvés)
--- Restaurant 5 ---

 Scraping Momidji ...
⏭️ Momidji ignoré ou erreur.
--- Restaurant 6 ---

 Scraping Krousty Sabaïdi ...
✅ Krousty Sabaïdi (45 plats trouvés)
--- Restaurant 7 ---

 Scraping O'Tacos ...
✅ O'Tacos (89 plats trouvés)
--- Restaurant 8 ---

 Scraping Chipotle Mexican Grill ...
✅ Chipotle Mexican Grill (39 plats trouvés)
--- Restaurant 9 ---

 Scraping KFC ...
✅ KFC (102 plats trouvés)

--- FINI ---
Total plats collectés : 456
['Pizza Primavera 🍕 (paris)', '25 Rue Saint-Marc, Paris, IDF 75002', '75002', 'Marguerita 🍅', '14,00\xa0€', '', 'Pizzas', '

In [ ]:
#Télécharger le fichier
from google.colab import files
files.download("asiatique.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Pour filitrer les restaurants dans Paris (750_ _)
import csv
import re

def filtrer_paris(input_csv, output_csv):
    lignes_conservees = []

    with open(input_csv, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            if len(row) < 3:
                continue

            code_postal_brut = row[2].strip()

            # Extraire un code postal à 5 chiffres
            match = re.search(r"\b(\d{5})\b", code_postal_brut)
            if not match:
                continue

            code_postal = match.group(1)

            # Filtrer uniquement les codes postaux de Paris
            if code_postal.startswith("750"):
                # Remplace la valeur de la colonne 3 par le code postal normalisé
                row[2] = code_postal
                lignes_conservees.append(row)

    # Écriture du nouveau fichier filtré
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(lignes_conservees)

    print(f"✅ Fichier filtré créé : {output_csv}")
    print(f"➡ {len(lignes_conservees)} lignes conservées (restaurants situés à Paris).")


# Exemple d'utilisation :
filtrer_paris("asiatique.csv", "asiatique_paris.csv")

In [ ]:
#TELECHARGER LE FICHIER
from google.colab import files
files.download("asiatique_paris.csv")